## 00 · Original MACCO four-loss baseline

Run top to bottom in a fresh GPU Colab runtime. This notebook prepares frozen COCO and Talk2Car manifests, performs a separate smoke and speed check, trains a five-epoch four-loss baseline from original OpenAI CLIP ViT-B/32 weights, and evaluates the resulting encoder. No results are pre-filled. The later variants use the saved manifests, settings, and baseline scores.

Pinned MACCO source: [revision 97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae](https://github.com/hiker-lw/MACCO/tree/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae). Model, tokenizer, data masking, losses, and scheduler below are copied or adapted in visible cells from that revision. The CLIP model and tokenizer originated with OpenAI (Copyright (c) 2021 OpenAI, MIT License). This notebook does not import the author repository or run an external training script.

### MACCO MIT license

```text
MIT License

Copyright (c) 2026 hiker-lw

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```


## S01 · Environment and assets

Install small dependencies; use Colab's matched PyTorch and torchvision. Download the CLIP BPE vocabulary and the original OpenAI ViT-B/32 checkpoint as data assets. Verify CUDA and the checkpoint SHA256.


In [ ]:
%pip -q install ftfy regex pandas pillow requests gdown huggingface_hub PyYAML tqdm


In [ ]:
from pathlib import Path
from types import SimpleNamespace
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import ast, csv, gc, hashlib, io, json, math, os, random, shutil, tarfile, time, zipfile
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_url, snapshot_download
import yaml

assert torch.cuda.is_available(), "Choose a GPU runtime in Colab."
DEVICE = torch.device("cuda")
SOURCE_REV = "97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae"
DATA_ROOT = Path("/content/macco_data")
DATASETS = DATA_ROOT / "datasets"
MANIFESTS = DATA_ROOT / "manifests"
BENCHMARKS = DATA_ROOT / "benchmarks"
INPUT_ROOT = Path("/content/macco_inputs")
RUN_ROOT = Path("/content/macco_runs")
for path in (DATASETS, MANIFESTS, BENCHMARKS, INPUT_ROOT, RUN_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def download_asset(url, path):
    path = Path(path)
    if path.is_file() and path.stat().st_size:
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".part")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(15, 120)) as response:
                response.raise_for_status()
                with temporary.open("wb") as output:
                    for chunk in response.iter_content(1 << 20):
                        if chunk:
                            output.write(chunk)
            temporary.replace(path)
            return path
        except Exception:
            temporary.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

BPE_PATH = download_asset(
    f"https://raw.githubusercontent.com/hiker-lw/MACCO/{SOURCE_REV}/src/open_clip_code/bpe_simple_vocab_16e6.txt.gz",
    DATA_ROOT / "bpe_simple_vocab_16e6.txt.gz",
)
CLIP_URL = "https://openaipublic.azureedge.net/clip/models/40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af/ViT-B-32.pt"
CLIP_PATH = download_asset(CLIP_URL, DATA_ROOT / "ViT-B-32-openai.pt")
CLIP_SHA256 = hashlib.sha256(CLIP_PATH.read_bytes()).hexdigest()
assert CLIP_SHA256.startswith("40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af")
print("Source revision:", SOURCE_REV)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("Original OpenAI CLIP SHA256:", CLIP_SHA256)


## S02 · Tokenizer

Pinned source: [src/open_clip_code/tokenizer.py](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/open_clip_code/tokenizer.py). The only adaptation is the explicit BPE data-asset path.


In [ ]:
""" CLIP tokenizer

Copied from https://github.com/openai/CLIP. Originally MIT License, Copyright (c) 2021 OpenAI.
"""
import gzip
import html
import os
from functools import lru_cache
from typing import Union, List

import ftfy
import regex as re
import torch


@lru_cache()
def default_bpe():
    return str(BPE_PATH)


@lru_cache()
def bytes_to_unicode():
    """
    Returns list of utf-8 byte and a corresponding list of unicode strings.
    The reversible bpe codes work on unicode strings.
    This means you need a large # of unicode characters in your vocab if you want to avoid UNKs.
    When you're at something like a 10B token dataset you end up needing around 5K for decent coverage.
    This is a signficant percentage of your normal, say, 32K bpe vocab.
    To avoid that, we want lookup tables between utf-8 bytes and unicode strings.
    And avoids mapping to whitespace/control characters the bpe code barfs on.
    """
    bs = list(range(ord("!"), ord("~")+1))+list(range(ord("¡"), ord("¬")+1))+list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8+n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))


def get_pairs(word):
    """Return set of symbol pairs in a word.
    Word is represented as tuple of symbols (symbols being variable-length strings).
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs


def basic_clean(text):
    text = ftfy.fix_text(text)
    text = html.unescape(html.unescape(text))
    return text.strip()


def whitespace_clean(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text


class SimpleTokenizer(object):
    def __init__(self, bpe_path: str = default_bpe(), special_tokens=None):
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}
        merges = gzip.open(bpe_path).read().decode("utf-8").split('\n')
        merges = merges[1:49152-256-2+1]
        merges = [tuple(merge.split()) for merge in merges]
        vocab = list(bytes_to_unicode().values())
        vocab = vocab + [v+'</w>' for v in vocab]
        for merge in merges:
            vocab.append(''.join(merge))
        if not special_tokens:
            special_tokens = ['<start_of_text>', '<end_of_text>']
        else:
            special_tokens = ['<start_of_text>', '<end_of_text>'] + special_tokens

        #vocab[-2]='<mask>'
        vocab.extend(special_tokens)
        self.encoder = dict(zip(vocab, range(len(vocab))))
        self.decoder = {v: k for k, v in self.encoder.items()}
        self.bpe_ranks = dict(zip(merges, range(len(merges))))
        self.cache = {t:t for t in special_tokens}
        special = "|".join(special_tokens)
        self.pat = re.compile(special + r"""|'s|'t|'re|'ve|'m|'ll|'d|[\p{L}]+|[\p{N}]|[^\s\p{L}\p{N}]+""", re.IGNORECASE)

        self.vocab_size = len(self.encoder)
        self.all_special_ids = [self.encoder[t] for t in special_tokens]

    def bpe(self, token):
        if token in self.cache:
            return self.cache[token]
        word = tuple(token[:-1]) + ( token[-1] + '</w>',)
        pairs = get_pairs(word)

        if not pairs:
            return token+'</w>'

        while True:
            bigram = min(pairs, key = lambda pair: self.bpe_ranks.get(pair, float('inf')))
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word)-1 and word[i+1] == second:
                    new_word.append(first+second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        word = ' '.join(word)
        self.cache[token] = word
        return word

    def encode(self, text):
        bpe_tokens = []
        text = whitespace_clean(basic_clean(text)).lower()
        for token in re.findall(self.pat, text):
            token = ''.join(self.byte_encoder[b] for b in token.encode('utf-8'))
            bpe_tokens.extend(self.encoder[bpe_token] for bpe_token in self.bpe(token).split(' '))
        return bpe_tokens

    def decode(self, tokens):
        text = ''.join([self.decoder[token] for token in tokens])
        text = bytearray([self.byte_decoder[c] for c in text]).decode('utf-8', errors="replace").replace('</w>', ' ')
        return text


_tokenizer = SimpleTokenizer()


def tokenize(texts: Union[str, List[str]], context_length: int = 77) -> torch.LongTensor:
    """
    Returns the tokenized representation of given input string(s)

    Parameters
    ----------
    texts : Union[str, List[str]]
        An input string or a list of input strings to tokenize
    context_length : int
        The context length to use; all CLIP models use 77 as the context length

    Returns
    -------
    A two-dimensional tensor containing the resulting tokens, shape = [number of input strings, context_length]
    """
    if isinstance(texts, str):
        texts = [texts]

    sot_token = _tokenizer.encoder["<start_of_text>"]
    eot_token = _tokenizer.encoder["<end_of_text>"]
    all_tokens = [[sot_token] + _tokenizer.encode(text) + [eot_token] for text in texts]
    result = torch.zeros(len(all_tokens), context_length, dtype=torch.long)

    for i, tokens in enumerate(all_tokens):
        if len(tokens) > context_length:
            tokens = tokens[:context_length]  # Truncate
            tokens[-1] = eot_token
        result[i, :len(tokens)] = torch.tensor(tokens)

    return result


## S03 · CLIP and MACCO model

Pinned sources: [CLIP model](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/open_clip_code/model.py), [MACCO wrapper](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/open_clip_code/MACCO_variant/MACCO_text_image.py), and [ViT-B/32 QuickGELU configuration](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/open_clip_code/model_configs/ViT-B-32-quickgelu.json). The wrapper method bindings are adjusted for current Python; encoder weights are converted to float32 before AMP.


In [ ]:
from collections import OrderedDict
from dataclasses import dataclass
from typing import Tuple, Union, Callable, Optional
from torch.utils.checkpoint import checkpoint
def to_2tuple(value):
    return value if isinstance(value, tuple) else (value, value)

class LayerNorm(nn.LayerNorm):
    """Subclass torch's LayerNorm to handle fp16."""

    def forward(self, x: torch.Tensor):
        orig_type = x.dtype
        x = F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        return x.to(orig_type)


class QuickGELU(nn.Module):
    # NOTE This is slower than nn.GELU or nn.SiLU and uses more GPU memory
    def forward(self, x: torch.Tensor):
        return x * torch.sigmoid(1.702 * x)


class ResidualAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, mlp_ratio: float = 4.0, act_layer: Callable = nn.GELU):
        super().__init__()

        self.attn = nn.MultiheadAttention(d_model, n_head)
        self.ln_1 = LayerNorm(d_model)
        mlp_width = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(OrderedDict([
            ("c_fc", nn.Linear(d_model, mlp_width)),
            ("gelu", act_layer()),
            ("c_proj", nn.Linear(mlp_width, d_model))
        ]))
        self.ln_2 = LayerNorm(d_model)

        self.attn_weight = None

    def attention(self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None):
        # return self.attn(x, x, x, need_weights=False, attn_mask=attn_mask)[0]
        attn_output, attn_weight = self.attn(x, x, x, need_weights=True, attn_mask=attn_mask)
        self.attn_weight = attn_weight
        return attn_output

    def forward(self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None):
        x = x + self.attention(self.ln_1(x), attn_mask=attn_mask)
        x = x + self.mlp(self.ln_2(x))
        return x



class Transformer(nn.Module):
    def __init__(self, width: int, layers: int, heads: int,  mlp_ratio: float = 4.0, act_layer: Callable = nn.GELU):
        super().__init__()
        self.width = width
        self.layers = layers
        self.grad_checkpointing = False

        self.resblocks = nn.ModuleList([
            ResidualAttentionBlock(width, heads, mlp_ratio, act_layer=act_layer)
            for _ in range(layers)
        ])

    def forward(self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None):
        for r in self.resblocks:
            if self.grad_checkpointing and not torch.jit.is_scripting():
                x = checkpoint(r, x, attn_mask)
            else:
                x = r(x, attn_mask=attn_mask)
        return x


class VisualTransformer(nn.Module):
    def __init__(
            self, image_size: int, patch_size: int, width: int, layers: int, heads: int, mlp_ratio: float,
            output_dim: int, act_layer: Callable = nn.GELU):
        super().__init__()
        self.image_size = to_2tuple(image_size)
        self.patch_size = to_2tuple(patch_size)
        self.grid_size = (self.image_size[0] // self.patch_size[0], self.image_size[1] // self.patch_size[1])
        self.output_dim = output_dim
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=width, kernel_size=patch_size, stride=patch_size, bias=False)

        scale = width ** -0.5
        self.class_embedding = nn.Parameter(scale * torch.randn(width))
        self.positional_embedding = nn.Parameter(scale * torch.randn(self.grid_size[0] * self.grid_size[1] + 1, width))
        self.ln_pre = LayerNorm(width)

        self.transformer = Transformer(width, layers, heads, mlp_ratio, act_layer=act_layer)

        self.ln_post = LayerNorm(width)
        self.proj = nn.Parameter(scale * torch.randn(width, output_dim))

    def lock(self, unlocked_groups=0, freeze_bn_stats=False):
        assert unlocked_groups == 0, 'partial locking not currently supported for this model'
        for param in self.parameters():
            param.requires_grad = False

    @torch.jit.ignore
    def set_grad_checkpointing(self, enable=True):
        self.transformer.grad_checkpointing = enable

    def forward(self, x: torch.Tensor):
        x = self.conv1(x)  # shape = [*, width, grid, grid]
        x = x.reshape(x.shape[0], x.shape[1], -1)  # shape = [*, width, grid ** 2]
        x = x.permute(0, 2, 1)  # shape = [*, grid ** 2, width]
        x = torch.cat(
            [self.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device),
             x], dim=1)  # shape = [*, grid ** 2 + 1, width]
        x = x + self.positional_embedding.to(x.dtype)
        x = self.ln_pre(x)

        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD

        x = self.ln_post(x[:, 0, :])

        if self.proj is not None:
            x = x @ self.proj

        return x


@dataclass
class CLIPVisionCfg:
    layers: Union[Tuple[int, int, int, int], int] = 12
    width: int = 768
    head_width: int = 64
    mlp_ratio: float = 4.0
    patch_size: int = 16
    image_size: Union[Tuple[int, int], int] = 224
    timm_model_name: str = None  # a valid model name overrides layers, width, patch_size
    timm_model_pretrained: bool = False  # use (imagenet) pretrained weights for named model
    timm_pool: str = 'avg'  # feature pooling for timm model ('abs_attn', 'rot_attn', 'avg', '')
    timm_proj: str = 'linear'  # linear projection for timm model output ('linear', 'mlp', '')


@dataclass
class CLIPTextCfg:
    context_length: int = 77
    vocab_size: int = 49408
    width: int = 512
    heads: int = 8
    layers: int = 12


class CLIP(nn.Module):
    def __init__(
            self,
            embed_dim: int,
            vision_cfg: CLIPVisionCfg,
            text_cfg: CLIPTextCfg,
            quick_gelu: bool = False,
    ):
        super().__init__()
        if isinstance(vision_cfg, dict):
            vision_cfg = CLIPVisionCfg(**vision_cfg)
        if isinstance(text_cfg, dict):
            text_cfg = CLIPTextCfg(**text_cfg)

        self.context_length = text_cfg.context_length

        # OpenAI models are pretrained w/ QuickGELU but native nn.GELU is both faster and more
        # memory efficient in recent PyTorch releases (>= 1.10).
        # NOTE: timm models always use native GELU regardless of quick_gelu flag.
        act_layer = QuickGELU if quick_gelu else nn.GELU

        if vision_cfg.timm_model_name:
            self.visual = TimmModel(
                vision_cfg.timm_model_name,
                pretrained=vision_cfg.timm_model_pretrained,
                pool=vision_cfg.timm_pool,
                proj=vision_cfg.timm_proj,
                embed_dim=embed_dim,
                image_size=vision_cfg.image_size
            )
            act_layer = nn.GELU  # so that text transformer doesn't use QuickGELU w/ timm models
        elif isinstance(vision_cfg.layers, (tuple, list)):
            vision_heads = vision_cfg.width * 32 // vision_cfg.head_width
            self.visual = ModifiedResNet(
                layers=vision_cfg.layers,
                output_dim=embed_dim,
                heads=vision_heads,
                image_size=vision_cfg.image_size,
                width=vision_cfg.width
            )
        else:
            vision_heads = vision_cfg.width // vision_cfg.head_width
            self.visual = VisualTransformer(
                image_size=vision_cfg.image_size,
                patch_size=vision_cfg.patch_size,
                width=vision_cfg.width,
                layers=vision_cfg.layers,
                heads=vision_heads,
                mlp_ratio=vision_cfg.mlp_ratio,
                output_dim=embed_dim,
                act_layer=act_layer,
            )

        self.transformer = Transformer(
            width=text_cfg.width,
            layers=text_cfg.layers,
            heads=text_cfg.heads,
            act_layer=act_layer,
        )

        self.vocab_size = text_cfg.vocab_size
        self.token_embedding = nn.Embedding(text_cfg.vocab_size, text_cfg.width)
        self.positional_embedding = nn.Parameter(torch.empty(self.context_length, text_cfg.width))
        self.ln_final = LayerNorm(text_cfg.width)

        self.text_projection = nn.Parameter(torch.empty(text_cfg.width, embed_dim))
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.register_buffer('attn_mask', self.build_attention_mask(), persistent=False)

        self.init_parameters()

    def init_parameters(self):
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.positional_embedding, std=0.01)
        nn.init.constant_(self.logit_scale, np.log(1 / 0.07))

        if hasattr(self.visual, 'init_parameters'):
            self.visual.init_parameters()

        proj_std = (self.transformer.width ** -0.5) * ((2 * self.transformer.layers) ** -0.5)
        attn_std = self.transformer.width ** -0.5
        fc_std = (2 * self.transformer.width) ** -0.5
        for block in self.transformer.resblocks:
            nn.init.normal_(block.attn.in_proj_weight, std=attn_std)
            nn.init.normal_(block.attn.out_proj.weight, std=proj_std)
            nn.init.normal_(block.mlp.c_fc.weight, std=fc_std)
            nn.init.normal_(block.mlp.c_proj.weight, std=proj_std)

        if self.text_projection is not None:
            nn.init.normal_(self.text_projection, std=self.transformer.width ** -0.5)

    def build_attention_mask(self):
        # lazily create causal attention mask, with full attention between the vision tokens
        # pytorch uses additive attention mask; fill with -inf
        mask = torch.empty(self.context_length, self.context_length)
        mask.fill_(float("-inf"))
        mask.triu_(1)  # zero out the lower diagonal
        return mask

    def lock_image_tower(self, unlocked_groups=0, freeze_bn_stats=False):
        # lock image tower as per LiT - https://arxiv.org/abs/2111.07991
        self.visual.lock(unlocked_groups=unlocked_groups, freeze_bn_stats=freeze_bn_stats)

    @torch.jit.ignore
    def set_grad_checkpointing(self, enable=True):
        self.visual.set_grad_checkpointing(enable)
        self.transformer.grad_checkpointing = enable

    def encode_image(self, image):
        return self.visual(image)

    def encode_text(self, text):
        x = self.token_embedding(text)  # [batch_size, n_ctx, d_model]

        x = x + self.positional_embedding
        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x, attn_mask=self.attn_mask)
        x = x.permute(1, 0, 2)  # LND -> NLD
        x = self.ln_final(x)

        # x.shape = [batch_size, n_ctx, transformer.width]
        # take features from the eot embedding (eot_token is the highest number in each sequence)
        x = x[torch.arange(x.shape[0]), text.argmax(dim=-1)] @ self.text_projection

        return x

    def forward(self, image, text):
        if image is None:
            return self.encode_text(text)
        elif text is None:
            return self.encode_image(image)
        image_features = self.encode_image(image)
        image_features = F.normalize(image_features, dim=-1)

        text_features = self.encode_text(text)
        text_features = F.normalize(text_features, dim=-1)

        return image_features, text_features, self.logit_scale.exp()



def convert_weights_to_fp16(model: nn.Module):
    """Convert applicable model parameters to fp16"""

    def _convert_weights_to_fp16(l):
        if isinstance(l, (nn.Conv1d, nn.Conv2d, nn.Linear)):
            l.weight.data = l.weight.data.half()
            if l.bias is not None:
                l.bias.data = l.bias.data.half()

        if isinstance(l, nn.MultiheadAttention):
            for attr in [*[f"{s}_proj_weight" for s in ["in", "q", "k", "v"]], "in_proj_bias", "bias_k", "bias_v"]:
                tensor = getattr(l, attr)
                if tensor is not None:
                    tensor.data = tensor.data.half()

        for name in ["text_projection", "proj"]:
            if hasattr(l, name):
                attr = getattr(l, name)
                if attr is not None:
                    attr.data = attr.data.half()

    model.apply(_convert_weights_to_fp16)


def build_model_from_openai_state_dict(state_dict: dict):
    vit = "visual.proj" in state_dict

    if vit:
        vision_width = state_dict["visual.conv1.weight"].shape[0]
        vision_layers = len(
            [k for k in state_dict.keys() if k.startswith("visual.") and k.endswith(".attn.in_proj_weight")])
        vision_patch_size = state_dict["visual.conv1.weight"].shape[-1]
        grid_size = round((state_dict["visual.positional_embedding"].shape[0] - 1) ** 0.5)
        image_size = vision_patch_size * grid_size
    else:
        counts: list = [
            len(set(k.split(".")[2] for k in state_dict if k.startswith(f"visual.layer{b}"))) for b in [1, 2, 3, 4]]
        vision_layers = tuple(counts)
        vision_width = state_dict["visual.layer1.0.conv1.weight"].shape[0]
        output_width = round((state_dict["visual.attnpool.positional_embedding"].shape[0] - 1) ** 0.5)
        vision_patch_size = None
        assert output_width ** 2 + 1 == state_dict["visual.attnpool.positional_embedding"].shape[0]
        image_size = output_width * 32

    embed_dim = state_dict["text_projection"].shape[1]
    context_length = state_dict["positional_embedding"].shape[0]
    vocab_size = state_dict["token_embedding.weight"].shape[0]
    transformer_width = state_dict["ln_final.weight"].shape[0]
    transformer_heads = transformer_width // 64
    transformer_layers = len(set(k.split(".")[2] for k in state_dict if k.startswith(f"transformer.resblocks")))

    vision_cfg = CLIPVisionCfg(
        layers=vision_layers,
        width=vision_width,
        patch_size=vision_patch_size,
        image_size=image_size,
    )
    text_cfg = CLIPTextCfg(
        context_length=context_length,
        vocab_size=vocab_size,
        width=transformer_width,
        heads=transformer_heads,
        layers=transformer_layers
    )
    model = CLIP(
        embed_dim,
        vision_cfg=vision_cfg,
        text_cfg=text_cfg,
        quick_gelu=True,  # OpenAI models were trained with QuickGELU
    )

    for key in ["input_resolution", "context_length", "vocab_size"]:
        state_dict.pop(key, None)

    convert_weights_to_fp16(model)
    model.load_state_dict(state_dict)
    return model.eval()


In [ ]:
import math
import random
from collections import OrderedDict
from dataclasses import dataclass
import logging
import sys
from typing import Tuple, Union, Callable, Optional
from torch.utils.checkpoint import checkpoint
from PIL import Image

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn


def get_2d_sincos_pos_embed(embed_dim, grid_size, cls_token=False):
    """
    grid_size: int of the grid height and width
    return:
    pos_embed: [grid_size*grid_size, embed_dim] or [1+grid_size*grid_size, embed_dim] (w/ or w/o cls_token)
    """
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)  # here w goes first
    grid = np.stack(grid, axis=0)

    grid = grid.reshape([2, 1, grid_size, grid_size])
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    if cls_token:
        pos_embed = np.concatenate([np.zeros([1, embed_dim]), pos_embed], axis=0)
    return pos_embed

def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0

    # use half of dimensions to encode grid_h
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])  # (H*W, D/2)
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])  # (H*W, D/2)

    emb = np.concatenate([emb_h, emb_w], axis=1) # (H*W, D)
    return emb

def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    """
    embed_dim: output dimension for each position
    pos: a list of positions to be encoded: size (M,)
    out: (M, D)
    """
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=float)
    omega /= embed_dim / 2.
    omega = 1. / 10000**omega  # (D/2,)

    pos = pos.reshape(-1)  # (M,)
    out = np.einsum('m,d->md', pos, omega)  # (M, D/2), outer product

    emb_sin = np.sin(out) # (M, D/2)
    emb_cos = np.cos(out) # (M, D/2)

    emb = np.concatenate([emb_sin, emb_cos], axis=1)  # (M, D)
    return emb

class ResidualCrossAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, mlp_ratio: float = 4.0, act_layer: Callable = nn.GELU):
        super().__init__()

        self.attn = nn.MultiheadAttention(d_model, n_head)
        self.ln_x = LayerNorm(d_model)
        self.ln_y = LayerNorm(d_model)
        mlp_width = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(OrderedDict([
            ("c_fc", nn.Linear(d_model, mlp_width)),
            ("gelu", act_layer()),
            ("c_proj", nn.Linear(mlp_width, d_model))
        ]))
        self.ln_post = LayerNorm(d_model)

        self.attn_weight = None

    def cross_attention(self, x: torch.Tensor, y: torch.Tensor, attn_mask: Optional[torch.Tensor] = None):
        CA_out = self.attn(x, y, y, need_weights=True, attn_mask=attn_mask)
        self.attn_weight = CA_out[1]
        return CA_out[0]

    def forward(self, x: torch.Tensor, y: torch.Tensor, attn_mask: Optional[torch.Tensor] = None):
        x = x + self.cross_attention(self.ln_x(x), self.ln_y(y), attn_mask=attn_mask)
        x = x + self.mlp(self.ln_post(x))
        return x


class Image_Predictor(nn.Module):
    def __init__(self, 
                 width: int, 
                 self_attn_layers: int,
                 cross_attn_layers: int,
                 heads: int,
                 patch_size: int = 32,
                 in_chans: int = 3
                 ):
        super().__init__()

        self.self_attn_layers = self_attn_layers
        self.cross_attn_layers = cross_attn_layers
        self.width = width

        self.num_patches = (224 // patch_size) ** 2
        self.decoder_embed = nn.Linear(width, width, bias=True)
        self.pos_embedding = nn.Parameter(torch.zeros(1, self.num_patches, width), requires_grad=False)

        self.self_attn_blocks = nn.ModuleList([
            ResidualAttentionBlock(width, heads)
            for _ in range(self.self_attn_layers)
        ])
        self.cross_attn_blocks = nn.ModuleList([
            ResidualCrossAttentionBlock(width, heads)
            for _ in range(self.cross_attn_layers)
        ])
        self.ln = nn.LayerNorm(width)
        
        self.patch_size = patch_size
        self.proj = nn.Sequential(
            nn.LayerNorm(width),
            nn.Linear(width, patch_size**2 * in_chans)  # Each token predicts pixels for one patch.
        )
        self._init_weights()
    
    def _init_weights(self):
        # Use the same initialization strategy as MAE.
        nn.init.xavier_uniform_(self.proj[-1].weight, gain=1/self.proj[-1].in_features**0.5)
        nn.init.zeros_(self.proj[-1].bias)

        pos_embedding = get_2d_sincos_pos_embed(self.pos_embedding.shape[-1], int(self.num_patches**.5), cls_token=False)
        self.pos_embedding.data.copy_(torch.from_numpy(pos_embedding).float().unsqueeze(0))

    def forward(self, x: torch.Tensor, y: torch.Tensor, x_cls: torch.Tensor, texts: torch.Tensor, attn_mask: Optional[torch.Tensor] = None):
        '''
        Input:
            x: masked image features (all tokens) from CLIP image encoder, [b, 49, 512]
            y: text features (all tokens) from CLIP text encoder, [b, 77, 512]
            texts: text tensor input to CLIP text encoder, [b, 77]
            attn_mask: full attention or causal attention mask between the tokens, default is None (full attention)
        Output:
            x: predicted pixel value, [b, 49, patch**2*3]
        '''
        y_cls = y[torch.arange(texts.shape[0]), texts.argmax(dim=-1)]
        y = (y + y_cls[:, None, :]) / 2

        x = (x + x_cls[:, None, :]) / 2

        x = self.decoder_embed(x)
        x = x + self.pos_embedding

        x = x.permute(1, 0, 2)
        y = y.permute(1, 0, 2)
        
        for self_attn_block in self.self_attn_blocks:
            x = self_attn_block(x)
        
        for cross_attn_block in self.cross_attn_blocks:
            x = cross_attn_block(x, y, attn_mask)
            
        x = x.permute(1, 0, 2)
        x = self.ln(x)

        x = self.proj(x)  # [B, num_patches, patch_size^2 * 3]

        return x

class Text_Predictor(nn.Module):
    def __init__(self, 
                 width: int, 
                 self_attn_layers: int,
                 cross_attn_layers: int,
                 heads: int
                 ):
        super().__init__()

        self.self_attn_layers = self_attn_layers
        self.cross_attn_layers = cross_attn_layers
        self.width = width
        self.self_attn_blocks = nn.ModuleList([
            ResidualAttentionBlock(width, heads)
            for _ in range(self.self_attn_layers)
        ])
        self.cross_attn_blocks = nn.ModuleList([
            ResidualCrossAttentionBlock(width, heads)
            for _ in range(self.cross_attn_layers)
        ])
        self.ln = nn.LayerNorm(width)
        # vocabulary size: 49408
        self.linear = nn.Linear(width, 49408, bias=True)
        self.softmax = nn.LogSoftmax(dim=-1)
        torch.nn.init.xavier_uniform_(self.linear.weight)
    
    def forward(self, x: torch.Tensor, y: torch.Tensor, y_cls: torch.Tensor, texts: torch.Tensor, attn_mask: Optional[torch.Tensor] = None):
        '''
        Input:
            x: masked text features (all tokens) from CLIP text encoder, [b, 77, 512]
            y: image features (all tokens) from CLIP image encoder, [b, 49, 512]
            texts: text tensor input to CLIP text encoder, [b, 77]
            attn_mask: full attention or causal attention mask between the tokens, default is None (full attention)
        Output:
            x: prediction probability of every word in the vocabulary, [b, 77, 49408]
        '''
        x_cls = x[torch.arange(texts.shape[0]), texts.argmax(dim=-1)]
        x = (x + x_cls[:, None, :]) / 2
        y = (y + y_cls[:, None, :]) / 2
        x = x.permute(1, 0, 2)
        y = y.permute(1, 0, 2)

        
        for self_attn_block in self.self_attn_blocks:
            x = self_attn_block(x)

        for cross_attn_block in self.cross_attn_blocks:
            x = cross_attn_block(x, y, attn_mask)
        
        x = x.permute(1, 0, 2)
        x = self.ln(x)
        x = self.linear(x) # classification 
        x = self.softmax(x)
        return x

class MACCO_CLIP_text_image(nn.Module):
    '''
    Cross-Modal Masked Composition Concept Modeling
    '''
    def __init__(self, CLIP, args, width: int=512, heads: int=8, output_dim: int=512):
        super(MACCO_CLIP_text_image, self).__init__()
        # pretrained CLIP model
        self.CLIP = CLIP
        self.args = args
        self.logit_scale = self.CLIP.logit_scale

        # replace encode_text, encode_image and forward function
        self.CLIP.encode_text = type(self).clip_encode_text.__get__(self.CLIP, type(self.CLIP))
        self.CLIP.encode_image = type(self).clip_encode_image.__get__(self.CLIP, type(self.CLIP))
        self.CLIP.visual.forward = type(self).clip_vit_forward.__get__(self.CLIP.visual, type(self.CLIP.visual))
        self.CLIP.forward = type(self).clip_forward.__get__(self.CLIP, type(self.CLIP))

        if self.args.model == "ViT-B-32":
            self.patch_size = 32
            width = 512
        elif self.args.model == "ViT-B-16":
            self.patch_size = 16
            width = 512
        elif self.args.model == "ViT-L-14":
            self.patch_size = 14
            width = 768

        # prediction layer for text
        self.prediction_text = Text_Predictor(width, args.text_predictor_self_attn_layers, args.text_predictor_cross_attn_layers, heads)
        # prediction layer for image
        self.prediction_image = Image_Predictor(width, args.image_predictor_self_attn_layers, args.image_predictor_cross_attn_layers, heads, patch_size=self.patch_size)

        # text mask token and image mask token, randomly initialized
        text_token_dim = CLIP.token_embedding.weight.shape[1]
        self.text_mask_token = nn.Parameter(torch.randn(text_token_dim))
        img_token_dim = CLIP.visual.positional_embedding.shape[1]
        self.img_mask_token = nn.Parameter(torch.randn(img_token_dim))
        self.init_parameters()

    def init_parameters(self):
        # code borrowed from neg_clip
        # init the mask token
        nn.init.normal_(self.text_mask_token, std=0.02)
        nn.init.normal_(self.img_mask_token, std=0.02)

        # Initialize parameters for prediction_text block
        if self.prediction_text.self_attn_layers > 0:
            self_attn_proj_std = (self.prediction_text.width ** -0.5) * ((2 * self.prediction_text.self_attn_layers) ** -0.5)
        if self.prediction_text.cross_attn_layers > 0:
            cross_attn_proj_std = (self.prediction_text.width ** -0.5) * ((2 * self.prediction_text.cross_attn_layers) ** -0.5)
        attn_std = self.prediction_text.width ** -0.5
        fc_std = (2 * self.prediction_text.width) ** -0.5
        for cross_attn_block in self.prediction_text.cross_attn_blocks:
            nn.init.normal_(cross_attn_block.attn.in_proj_weight, std=attn_std)
            nn.init.normal_(cross_attn_block.attn.out_proj.weight, std=cross_attn_proj_std)
            nn.init.normal_(cross_attn_block.mlp.c_fc.weight, std=fc_std)
            nn.init.normal_(cross_attn_block.mlp.c_proj.weight, std=cross_attn_proj_std)
        
        for self_attn_block in self.prediction_text.self_attn_blocks:
            nn.init.normal_(self_attn_block.attn.in_proj_weight, std=attn_std)
            nn.init.normal_(self_attn_block.attn.out_proj.weight, std=self_attn_proj_std)
            nn.init.normal_(self_attn_block.mlp.c_fc.weight, std=fc_std)
            nn.init.normal_(self_attn_block.mlp.c_proj.weight, std=self_attn_proj_std)
        
        # Initialize parameters for prediction_image block
        if self.prediction_image.self_attn_layers > 0:
            self_attn_proj_std = (self.prediction_image.width ** -0.5) * ((2 * self.prediction_image.self_attn_layers) ** -0.5)
        if self.prediction_image.cross_attn_layers > 0:
            cross_attn_proj_std = (self.prediction_image.width ** -0.5) * ((2 * self.prediction_image.cross_attn_layers) ** -0.5)
        attn_std = self.prediction_image.width ** -0.5
        fc_std = (2 * self.prediction_image.width) ** -0.5
        for cross_attn_block in self.prediction_image.cross_attn_blocks:
            nn.init.normal_(cross_attn_block.attn.in_proj_weight, std=attn_std)
            nn.init.normal_(cross_attn_block.attn.out_proj.weight, std=cross_attn_proj_std)
            nn.init.normal_(cross_attn_block.mlp.c_fc.weight, std=fc_std)
            nn.init.normal_(cross_attn_block.mlp.c_proj.weight, std=cross_attn_proj_std)
        
        for self_attn_block in self.prediction_image.self_attn_blocks:
            nn.init.normal_(self_attn_block.attn.in_proj_weight, std=attn_std)
            nn.init.normal_(self_attn_block.attn.out_proj.weight, std=self_attn_proj_std)
            nn.init.normal_(self_attn_block.mlp.c_fc.weight, std=fc_std)
            nn.init.normal_(self_attn_block.mlp.c_proj.weight, std=self_attn_proj_std)
    
    def clip_encode_text(self, text, text_mask_index, text_mask_token):
        # makesure text_mask_index is bool type
        text_mask_index = text_mask_index.to(dtype=torch.bool)

        # initialize the key mask matrix (dim: [2N,L])
        key_mask = torch.zeros([text.shape[0] * 2, text.shape[1]], dtype=torch.bool, device=text.device)
        
        # fill the lower half of the key mask matrix with the generated mask index
        key_mask[text.shape[0]:] = text_mask_index

        x_full_mask = torch.cat([text, text], dim=0) # 2N * L * D

        x_full_mask = self.token_embedding(x_full_mask)  # [batch_size, n_ctx, d_model]
        
        # fill the masked position with mask_token
        # x_full_and_masked[key_mask] = mask_token[None, :]
        x_full_mask = x_full_mask * (~key_mask[:, :, None]) + \
                text_mask_token[None, None, :] * key_mask[:, :, None]

        x_full_mask = x_full_mask + self.positional_embedding
        x_full_mask = x_full_mask.permute(1, 0, 2)  # NLD -> LND
        x_full_mask = self.transformer(x_full_mask, attn_mask=self.attn_mask)
        x_full_mask = x_full_mask.permute(1, 0, 2)  # LND -> NLD
        x_full_mask = self.ln_final(x_full_mask)

        # projection
        # x_full_and_masked.shape = [batch_size * 2 , n_ctx, transformer.width]
        # take features from the eot embedding (eot_token is the highest number in each sequence)
        eot_token_idx = text.argmax(dim=-1)
        x_full_mask_proj = x_full_mask @ self.text_projection

        # all text token feature
        x_all_token = x_full_mask_proj[0:text.shape[0]]
        x_masked_all_token = x_full_mask_proj[text.shape[0]: text.shape[0]*2]

        # take CLS token from the eot position
        x = x_all_token[torch.arange(text.shape[0]), eot_token_idx]
        x_masked = x_masked_all_token[torch.arange(text.shape[0]), eot_token_idx]

        return x, x_all_token, x_masked, x_masked_all_token

    def clip_encode_image(self, image, image_mask_index, img_mask_token):
        return self.visual(image, image_mask_index, img_mask_token)

    def clip_vit_forward(self, x, image_mask_index, img_mask_token):
        x = self.conv1(x)  # shape = [*, width, grid, grid]
        x = x.reshape(x.shape[0], x.shape[1], -1)  # shape = [*, width, grid ** 2]
        x = x.permute(0, 2, 1)  # shape = [*, grid ** 2, width]
        x = torch.cat(
            [self.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device),
             x], dim=1)  # shape = [*, grid ** 2 + 1, width]
        
        # initialize the key mask matrix (dim: [2N, L])
        key_mask = torch.zeros([x.shape[0] * 2, x.shape[1]], dtype=torch.bool, device=x.device)
        
        # fill the lower half of the key mask matrix with the generated mask index
        key_mask[x.shape[0]:] = image_mask_index

        x_full_mask = torch.cat([x, x], dim=0) # 2N * L * D

        # fill the masked position with mask_token
        x_full_mask = x_full_mask * (~key_mask[:, :, None]) + \
                img_mask_token[None, None, :] * key_mask[:, :, None]
        
        x_full_mask = x_full_mask + self.positional_embedding.to(x_full_mask.dtype)
        x_full_mask = self.ln_pre(x_full_mask)

        x_full_mask = x_full_mask.permute(1, 0, 2)  # NLD -> LND
        x_full_mask = self.transformer(x_full_mask)
        x_full_mask = x_full_mask.permute(1, 0, 2)  # LND -> NLD

        x_full_mask = self.ln_post(x_full_mask) # modified here to output both cls token features and image patch features

        if self.proj is not None:
            x_full_mask_proj = x_full_mask @ self.proj
            cls_token = x_full_mask_proj[:x.shape[0], 0]
            cls_token_masked = x_full_mask_proj[x.shape[0]:, 0]
            patch_token = x_full_mask_proj[:x.shape[0], 1:]
            patch_token_masked = x_full_mask_proj[x.shape[0]:, 1:]

        return cls_token, patch_token, cls_token_masked, patch_token_masked
    
    def clip_forward(self, image, text, image_mask_index=None, img_mask_token=None, text_mask_index=None, text_mask_token=None,):
        if image is None:
            self.encode_text(text, text_mask_index, text_mask_token)
        elif text is None:
            return self.encode_image(image, image_mask_index, img_mask_token)

        # get text and image feature
        text_feature, text_feature_all, masked_text_feature, masked_text_feature_all = self.encode_text(text, text_mask_index, text_mask_token)
        image_feature, image_feature_all, masked_image_feature, masked_image_feature_all = self.encode_image(image, image_mask_index, img_mask_token)

        image_feature_before_norm = image_feature
        masked_image_feature_before_norm = masked_image_feature

        text_feature = F.normalize(text_feature, dim=-1)
        image_feature = F.normalize(image_feature, dim=-1)
        masked_text_feature = F.normalize(masked_text_feature, dim=-1)
        masked_image_feature = F.normalize(masked_image_feature, dim=-1)


        result_dict = {
            "image_feature": image_feature,
            "text_feature": text_feature,
            "image_feature_before_norm": image_feature_before_norm,
            "masked_image_feature_before_norm": masked_image_feature_before_norm,
            "image_feature_all": image_feature_all,
            "text_feature_all": text_feature_all,
            "masked_text_feature": masked_text_feature,
            "masked_text_feature_all": masked_text_feature_all,
            "masked_image_feature": masked_image_feature,
            "masked_image_feature_all": masked_image_feature_all,
            "logit_scale": self.logit_scale.exp(),
        }

        return result_dict
    
    def forward(self, images, texts, image_mask_index, text_mask_index):
        '''
        Input:
            images: image tensor input to CLIP image encoder, [b, 3, 224, 224]
            texts: text tensor input to CLIP text encoder, [b, 77]
            text_mask_index: mask for text, every item is a boolean value (True: mask, False: not mask), [b, 77]
            image_mask_index: mask for image, every item is a boolean value (True: mask, False: not mask), [b, 50]
        Output:
            output: a dictionary
            output["img_feat_cls_token"]: image CLS token from CLIP image encoder, [b, 512]
            output["img_feat_all_token"]: image all tokens from CLIP image encoder, [b, 49, 512]
            output["text_feat_cls_token"]: text CLS token from CLIP text encoder, [b, 512]
            output["text_feat_all_token"]: text all tokens from CLIP text encoder, [b, 77, 512]
            output["masked_text_feat_cls_token"]: masked text CLS token from CLIP text encoder, [b, 512]
            output["masked_text_feat_all_token"]: masked text all tokens from CLIP text encoder, [b, 77, 512]
            output["pred_text"]: prediction probability of every word in the vocabulary, [b, 77, 49408]
            output["target_text"]: original input text, [b, 77]
            output["masked_image_feat_cls_token"]: masked image CLS token from CLIP image encoder, [b, 512]
            output["masked_image_feat_all_token"]: masked image all tokens from CLIP image encoder, [b, 49, 512]
            output["pred_image"]: predict patch pixel value, [b, 49, patch_size**2*3]
            output["target_image"]: original input image, [b, 3, 224, 224]
            output["images"]: input image tensor, [b, 3, 224, 224]
            output["texts"]: input text tensor, [b, 77]
            output["logit_scale"]: 1/temperature
            output["image_mask_index"]: mask for image, every item is a boolean value (True: mask, False: not mask), [b, 50]
        '''

        # output from CLIP
        CLIP_out = self.CLIP(images, texts, image_mask_index, self.img_mask_token, text_mask_index, self.text_mask_token)
        image_feature = CLIP_out["image_feature"]
        text_feature = CLIP_out["text_feature"]
        image_feature_before_norm = CLIP_out["image_feature_before_norm"]
        masked_image_feature_before_norm = CLIP_out["masked_image_feature_before_norm"]
        image_feature_all = CLIP_out["image_feature_all"]
        text_feature_all = CLIP_out["text_feature_all"]
        masked_text_feature = CLIP_out["masked_text_feature"]
        masked_text_feature_all = CLIP_out["masked_text_feature_all"]
        masked_image_feature = CLIP_out["masked_image_feature"]
        masked_image_feature_all = CLIP_out["masked_image_feature_all"]
        logit_scale = CLIP_out["logit_scale"]

        # predict the text
        pred_text = self.prediction_text(masked_text_feature_all, image_feature_all.detach(), image_feature_before_norm.detach(), texts)
        target_text = texts

        # predict the image
        pred_image = self.prediction_image(masked_image_feature_all.detach(), text_feature_all, masked_image_feature_before_norm.detach(), texts)
        target_image = images

        # normalization
        image_feature_all = F.normalize(image_feature_all, dim=-1)
        text_feature_all = F.normalize(text_feature_all, dim=-1)

        logit_scale = self.logit_scale.exp()

        output = {"img_feat_cls_token": image_feature,
                  "img_feat_all_token": image_feature_all,
                  "text_feat_cls_token": text_feature,
                  "text_feat_all_token": text_feature_all,

                  "masked_text_feat_cls_token": masked_text_feature,
                  "masked_text_feat_all_token": masked_text_feature_all,

                  "pred_text": pred_text,
                  "target_text": target_text,

                  "masked_image_feat_cls_token": masked_image_feature,
                  "masked_image_feat_all_token": masked_image_feature_all,

                  "pred_image": pred_image,
                  "target_image": target_image,

                  "images": images,
                  "texts":texts,

                  "logit_scale": logit_scale,
                  "image_mask_index": image_mask_index,
                  "text_mask_index": text_mask_index}

        return output


if __name__ == '__main__':
     pass


In [ ]:
def load_original_clip():
    try:
        checkpoint = torch.jit.load(str(CLIP_PATH), map_location="cpu")
        state = checkpoint.state_dict()
        del checkpoint
    except RuntimeError:
        state = torch.load(CLIP_PATH, map_location="cpu", weights_only=True)
        if "state_dict" in state:
            state = state["state_dict"]
    clip = build_model_from_openai_state_dict(state)
    clip.float()
    assert isinstance(clip.visual, VisualTransformer)
    assert clip.visual.patch_size == (32, 32)
    assert any(isinstance(module, QuickGELU) for module in clip.modules())
    return clip

def new_macco(config):
    random.seed(config["seed"])
    np.random.seed(config["seed"])
    torch.manual_seed(config["seed"])
    clip = load_original_clip()
    args = SimpleNamespace(**config)
    model = MACCO_CLIP_text_image(clip, args).float().to(DEVICE)
    assert all(param.requires_grad for param in model.CLIP.parameters())
    return model


## S04 · Preprocessing and dataset

Pinned sources: [training utilities](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/training/utils.py) and [joint image–text dataset](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/training/data_macco.py). The dataset constructor is adapted to read a supplied manifest and staged image root; masking and row logic remain from the author.


In [ ]:
import torch
from torchvision.transforms import functional as F
from torchvision.transforms import Normalize, Compose, RandomResizedCrop, InterpolationMode, ToTensor, Resize, \
    CenterCrop

class RandomResizedCrop_with_tracking(RandomResizedCrop):
    """RandomResizedCrop variant that records the sampled transform parameters."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.last_params = None  # Store the latest transform parameters.

    def forward(self, img):
        # Sample crop parameters.
        i, j, h, w = self.get_params(img, self.scale, self.ratio)
        self.last_params = (i, j, h, w, img.height, img.width)
        
        return F.resized_crop(
            img, i, j, h, w, self.size, self.interpolation
        )

class BBoxAdapter:
    def __init__(self, image_size=224):
        self.image_size = image_size
        
    def transform_bbox(self, orig_bbox, transform_params):
        """
        Transform a bounding box according to the sampled augmentation parameters.
        :param orig_bbox: [x1,y1,x2,y2] coordinates in the original image
        :param transform_params: (i,j,h,w, orig_h, orig_w)
        :return: transformed coordinates [x1,y1,x2,y2]
        """
        i, j, h, w, orig_h, orig_w = transform_params
        # Compute scaling factors.
        scale_h = self.image_size / h
        scale_w = self.image_size / w
        
        # Convert coordinates from the original image to the crop, then resize.
        x1 = max(orig_bbox[0] - j, 0) * scale_w
        y1 = max(orig_bbox[1] - i, 0) * scale_h
        x2 = min(orig_bbox[2] - j, w) * scale_w
        y2 = min(orig_bbox[3] - i, h) * scale_h
        
        return [
            x1, y1,
            x2, y2
        ]

def _convert_to_rgb(image):
    return image.convert('RGB')
def create_transform_with_tracking(image_size=224):
    crop = RandomResizedCrop_with_tracking(
        image_size, 
        scale=(0.9, 1.0), 
        interpolation=InterpolationMode.BICUBIC
    )
    mean = (0.48145466, 0.4578275, 0.40821073)  # OpenAI dataset mean
    std = (0.26862954, 0.26130258, 0.27577711)  # OpenAI dataset std
    def get_transform_params():
        return crop.last_params
    
    transforms = Compose([
        crop,
        _convert_to_rgb,
        ToTensor(),
        Normalize(mean, std)
    ])
    
    return transforms, get_transform_params

def bbox_to_patch_mask(transformed_bboxes, image_size=224, patch_size=32):
    """
    Map transformed bounding boxes to a ViT patch mask.
    :param transformed_bboxes: list of transformed [x1,y1,x2,y2] coordinates
    :param image_size: input image size
    :param patch_size: patch size
    :return: mask_flag (Tensor [n_tokens]) 
    """
    # Compute grid parameters.
    grid_size = image_size // patch_size
    n_patches = grid_size ** 2
    
    # Initialize mask, including the class-token position.
    mask_flag = torch.zeros(n_patches + 1, dtype=torch.bool)
    
    for bbox in transformed_bboxes:
        x1, y1, x2, y2 = bbox
        
        # Compute the covered patch range.
        col_start = int(x1 // patch_size)
        col_end = int((x2 - 1e-5) // patch_size)  # Avoid floating-point boundary spillover.
        row_start = int(y1 // patch_size)
        row_end = int((y2 - 1e-5) // patch_size)
        
        # Clamp to valid patch coordinates.
        col_start = max(0, min(col_start, grid_size-1))
        col_end = max(0, min(col_end, grid_size-1))
        row_start = max(0, min(row_start, grid_size-1))
        row_end = max(0, min(row_end, grid_size-1))
        
        # Fill the mask region.
        for row in range(row_start, row_end+1):
            for col in range(col_start, col_end+1):
                # ViT patch tokens are flattened in row-major order.
                patch_idx = row * grid_size + col
                mask_flag[patch_idx + 1] = True  # +1 skips the class token.

    return mask_flag


In [ ]:
import logging
class CsvDataset_mask_text_image(Dataset):
    """
    Mask compositional concepts in image and text.
    """
    def __init__(self, args, input_filename, transforms, get_transform_params, data_root, img_key="filepath", caption_key="title", sep="\t"):
        self.args = args
        df = pd.read_csv(input_filename, sep=sep, converters={
            "object_list": ast.literal_eval, "relation_list": ast.literal_eval,
            "attribute_list": ast.literal_eval, "relation_phrase_list": ast.literal_eval,
            "attribute_phrase_list": ast.literal_eval,
            "relation_attribute_phrase_bbox_list": ast.literal_eval,
            "neg_caption": ast.literal_eval, "neg_image": ast.literal_eval,
        })
        self.data_path = str(data_root)
        self.images = df[img_key].tolist()
        self.captions = df[caption_key].tolist()
        self.hard_captions = df["neg_caption"].tolist()
        self.noun_list = df["object_list"].tolist()
        self.relation_word_list = df["relation_list"].tolist()
        self.adjective_word_list = df["attribute_list"].tolist()
        self.relation_attribute_phrase_bbox_list = df["relation_attribute_phrase_bbox_list"].tolist()
        self.transforms = transforms
        self.get_transform_params = get_transform_params
        self.bbox_adapter = BBoxAdapter(224)

    def __len__(self):
        return len(self.captions)

    def get_word_token_pos_in_text_token(self, word, text):
        # get the word token position in all text token
        text_token_list = text.clone().tolist()
        word_token = _tokenizer.encode(word)
        start_idx = -1
        for idx in range(len(text)):
            if text_token_list[idx: idx + len(word_token)] == word_token:
                start_idx = idx
                break
        if start_idx != -1:
            return [start_idx + i for i in range(len(word_token))]
        else:
            # raise Exception("Not found matched word in the text!!!")
            return []

    def get_text_mask_token_flag(self, idx, texts):
        comp_concept_token_idx = [] # Indices of relation and attribute tokens in the full caption.
        object_token_idx = [] # Indices of object tokens in the full caption.
        attribute_word_list = self.adjective_word_list[idx]
        relation_word_list = self.relation_word_list[idx]
        object_word_list = self.noun_list[idx]
        
        for word in relation_word_list:
            # if word.lower() != "the" and word.lower() != "in" and word.lower() != "on":
            if word.lower() != "the":
                comp_concept_token_idx += self.get_word_token_pos_in_text_token(word, texts)
        
        for word in attribute_word_list:
            if word.lower() != "the":
                comp_concept_token_idx += self.get_word_token_pos_in_text_token(word, texts)

        for word in object_word_list:
            if word.lower() != "the":
                object_token_idx += self.get_word_token_pos_in_text_token(word, texts)
        
        comp_concept_token_flag = torch.zeros(len(texts)).bool()
        comp_concept_token_flag[comp_concept_token_idx] = True

        object_token_flag = torch.zeros(len(texts)).bool()
        object_token_flag[object_token_idx] = True

        return comp_concept_token_flag, object_token_flag

    def get_image_mask_token_flag(self, original_bbox_list, transform_params):
        if self.args.model == "ViT-B-32":
            patch_size = 32
            image_token_num = 49
        elif self.args.model == "ViT-B-16":
            patch_size = 16
            image_token_num = 196
        elif self.args.model == "ViT-L-14":
            patch_size = 14
            image_token_num = 256

        if not original_bbox_list:
            image_mask_token_flag = torch.zeros(image_token_num + 1, dtype=torch.bool)
        else:
            transformed_bbox_list = [
                self.bbox_adapter.transform_bbox(b, transform_params) 
                for b in original_bbox_list
            ]
            image_mask_token_flag = bbox_to_patch_mask(transformed_bbox_list, image_size=224, patch_size=patch_size)

        return image_mask_token_flag

    def generate_random_mask_image(self, mask_ratio=0.6):
        if self.args.model == "ViT-B-32":
            image_token_num = 49
        elif self.args.model == "ViT-B-16":
            image_token_num = 196
        elif self.args.model == "ViT-L-14":
            image_token_num = 256
        mask = torch.zeros(image_token_num + 1, dtype=torch.bool)
        n = mask.numel() - 1  # Number of image tokens.
        num_mask = int(round(n * mask_ratio))
        num_mask = max(0, min(num_mask, n))  # Keep the value in range.
        
        if num_mask > 0:
            # Generate random indices and set them to True.
            indices = torch.randperm(n)[:num_mask]
            mask.view(-1)[indices + 1] = True
        return mask

    def generate_random_rectangular_mask_image(self, mask_ratio=0.6):
        # Determine the grid size from the model type.
        if self.args.model == "ViT-B-32":
            grid_h, grid_w = 7, 7   # 7x7 grid.
        elif self.args.model == "ViT-B-16":
            grid_h, grid_w = 16, 16 # 16x16 grid.
        elif self.args.model == "ViT-L-14":
            grid_h, grid_w = 14, 14 # 14x14 grid.
        else:
            raise ValueError(f"Unsupported model: {self.args.model}")
        
        total_patches = grid_h * grid_w
        mask = torch.zeros(total_patches + 1, dtype=torch.bool)  # Include the class-token position.
        
        # Handle boundary cases.
        if mask_ratio <= 0:
            return mask
        elif mask_ratio >= 1:
            mask[1:] = True  # Mask all patches except the class token.
            return mask
        
        # Compute the rounded target mask area.
        target_area = int(round(total_patches * mask_ratio))
        target_area = max(1, min(target_area, total_patches))  # Keep the value in range.
        
        # Randomly generate rectangle parameters.
        while True:
            # Random rectangle height and width, at least one cell each.
            h = random.randint(1, grid_h)
            w = random.randint(1, grid_w)
            
            # Compute the current rectangle area.
            current_area = h * w
            
            # Accept rectangles close to the target area, with +/-25% tolerance.
            if abs(current_area - target_area) <= max(1, 0.25 * target_area):
                break
        
        # Randomly select the rectangle start position.
        top = random.randint(0, grid_h - h)
        left = random.randint(0, grid_w - w)
        
        # Create the grid mask.
        grid_mask = torch.zeros((grid_h, grid_w), dtype=torch.bool)
        # Set the rectangle region to True.
        grid_mask[top:top+h, left:left+w] = True
        
        # Flatten the grid mask and apply it to the output.
        mask[1:1+total_patches] = grid_mask.flatten()
        
        return mask

    def generate_random_mask_text(self, mask_ratio=0.15, texts=None):
        text_token_num = 77
        comp_concept_token_flag = torch.zeros(text_token_num, dtype=torch.bool)
        object_token_flag = torch.zeros(text_token_num, dtype=torch.bool)

        # Get the end-of-text token position. This path operates on a single sample.
        end_pos = texts.argmax(dim=-1).squeeze().item()  # Convert to a Python scalar.

        # Content tokens range from after the start token to before the end token.
        content_start = 1  # First position after the start-of-text token.
        content_end = end_pos  # End-of-text position, exclusive.

        # Valid content positions: [1, 2, ..., end_pos - 1].
        content_indices = torch.arange(content_start, content_end)
        n = len(content_indices)  # Number of content tokens.

        # Compute the number of tokens to mask.
        num_mask = int(round(n * mask_ratio))
        num_mask = max(0, min(num_mask, n))  # Keep the value in range.

        # Generate an independently sampled compositional-concept mask.
        if num_mask > 0:
            selected = torch.randperm(n)[:num_mask]  # Randomly select within the content range.
            comp_concept_token_flag[content_indices[selected]] = True

        # Generate an independently sampled object mask.
        if num_mask > 0:
            selected = torch.randperm(n)[:num_mask]  # Use a fresh random selection.
            object_token_flag[content_indices[selected]] = True

        return comp_concept_token_flag, object_token_flag

    def __getitem__(self, idx):

        images = self.transforms(Image.open(os.path.join(self.data_path, str(self.images[idx]))))
        transform_params = self.get_transform_params()

        if self.relation_attribute_phrase_bbox_list[idx]:
            original_bbox_list = random.sample(list(self.relation_attribute_phrase_bbox_list[idx][0].values()), k=1)[0]
        else:
            original_bbox_list = []

        caption = self.captions[idx]
        caption = caption.strip()
        texts = tokenize([str(caption)])[0]
        try:
            chosen_caption = random.choice(self.hard_captions[idx])
            hard_captions = tokenize([str(chosen_caption)])[0] # hard negative caption for texts
        except:
            chosen_caption = "A photo of full black image."
            hard_captions = tokenize([str(chosen_caption)])[0]

        comp_concept_token_flag, object_token_flag = self.get_text_mask_token_flag(idx, texts)       
        
        if random.random() < self.args.random_rectangular_mask_prob:
            image_mask_token_flag = self.generate_random_rectangular_mask_image(mask_ratio=self.args.random_rectangular_mask_image_ratio)
        else:
            image_mask_token_flag = self.get_image_mask_token_flag(original_bbox_list, transform_params)

        if self.args.random_mask_text:
            comp_concept_token_flag, object_token_flag = self.generate_random_mask_text(mask_ratio=self.args.random_mask_text_ratio, texts=texts)        
        
        if self.args.random_mask_image:
            if self.args.random_rectangular_mask_image:
                image_mask_token_flag = self.generate_random_rectangular_mask_image(mask_ratio=self.args.random_rectangular_mask_image_ratio)
            else:
                image_mask_token_flag = self.generate_random_mask_image(mask_ratio=self.args.random_mask_image_ratio)

        return images, texts, hard_captions, comp_concept_token_flag, object_token_flag, image_mask_token_flag


## S05 · Four author losses

Pinned source: [MACCO joint loss](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/open_clip_code/loss_fn/MACCO_CLIP_loss_text_image_raw_space.py). MCA, MIR, MLM, and MIM use the author target construction and reductions.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_contrast_loss(image_features,
                                 text_features,
                                 logit_scale,
                                 device):
    num_logits = image_features.shape[0] # batch size
    # normal contrastive loss
    
    logits_per_img_to_text = logit_scale * image_features @ text_features.T
    logits_per_text_to_img = logit_scale * text_features @ image_features.T
    
    labels = torch.arange(num_logits, device=device, dtype=torch.long)
    normal_loss = (F.cross_entropy(logits_per_img_to_text, labels) + \
                F.cross_entropy(logits_per_text_to_img, labels))/2
    
    return normal_loss


def compute_contrast_loss_with_negative_text_image(image_features, text_features, logit_scale, device):
    num_logits = int(text_features.shape[0] // 2) # batch size
    # normal contrastive loss with negative text and image, compute the contrastive loss of negative image to text, 
    # and the contrastive loss of negative text to image
    
    logits_per_text_to_img = logit_scale * text_features @ image_features.T
    logits_per_img_to_text = logit_scale * image_features @ text_features.T
    
    labels_text_to_img = torch.arange(num_logits, device=device, dtype=torch.long).repeat(2)
    labels_img_to_text = torch.arange(num_logits, device=device, dtype=torch.long).repeat(2)
    normal_loss = (F.cross_entropy(logits_per_img_to_text, labels_img_to_text) + \
                F.cross_entropy(logits_per_text_to_img, labels_text_to_img))/2
    
    return normal_loss


def patchify(imgs: torch.Tensor, patch_size: int):
        """
        Split images into patches, following the official MAE implementation.
        imgs: [B, 3, H, W]
        returns: [B, num_patches, patch_size^2 * 3]
        """
        p = patch_size
        assert imgs.shape[2] == imgs.shape[3] and imgs.shape[2] % p == 0

        h = w = imgs.shape[2] // p
        x = imgs.reshape(shape=(imgs.shape[0], 3, h, p, w, p))
        x = torch.einsum('nchpwq->nhwpqc', x)
        x = x.reshape(shape=(imgs.shape[0], h * w, p**2 * 3))
        return x


class macco_clip_loss_text_image(nn.Module):
    # predict raw text signals, compute masked text reconstruction loss per sample, then average across batch.
    # predict raw pixel signals, compute masked image reconstruction loss per sample, then average across batch.
    # treat cls token of masked text and masked image as negative samples, 
    # also compute the contrastive loss of negative text to image and the contrastive loss of negative image to text
    # add the contrastive loss bewteen masked text and text, masked image and image (intra-modal contrastive loss)
    def __init__(self, args, norm_pix_loss: bool = True):
        super().__init__()
        self.norm_pix_loss = norm_pix_loss
        self.mse_loss = nn.MSELoss(reduction='none')
        self.args = args
        if self.args.model == "ViT-B-32":
            self.patch_size = 32
        elif self.args.model == "ViT-B-16":
            self.patch_size = 16
        elif self.args.model == "ViT-L-14":
            self.patch_size = 14
    
    def get_loss_keys(self):
        return ["contrast_loss", "intra_modal_contrast_loss", "mask_loss_text", "mask_loss_image", "total_loss"]
    
    def _compute_contrast_loss(self, image_features, text_features, masked_image_features, masked_text_features, logit_scale):
        # normal contrastive loss
        device = image_features.device
        text_features = torch.cat([text_features, masked_text_features], dim=0)
        image_features = torch.cat([image_features, masked_image_features], dim=0)
        contrast_loss = compute_contrast_loss_with_negative_text_image(image_features, text_features, logit_scale, device)
        return contrast_loss
    
    def _compute_intra_modal_contrast_loss(self, image_features, text_features, masked_image_features, masked_text_features, logit_scale):
        # intra-modal contrastive loss
        device = image_features.device
        text_intra_modal_contrast_loss = compute_contrast_loss(masked_text_features, text_features, logit_scale, device)
        image_intra_modal_contrast_loss = compute_contrast_loss(masked_image_features, image_features, logit_scale, device)
        intra_modal_contrast_loss = 1/2 * (text_intra_modal_contrast_loss + image_intra_modal_contrast_loss)
        return intra_modal_contrast_loss
    
    def _compute_mask_loss_image(self, pred_image, target_image, image_mask_index, patch_size=32):
        image_mask_index = image_mask_index[:, 1:].to(pred_image.device)
        target_patches = patchify(target_image, patch_size)
        
        if self.norm_pix_loss:
            mean = target_patches.mean(dim=-1, keepdim=True)
            var = target_patches.var(dim=-1, keepdim=True)
            target_patches = (target_patches - mean) / (var + 1e-6).sqrt()

        pred_image = pred_image.reshape(target_patches.shape[0], target_patches.shape[1], target_patches.shape[2])
        mask_loss_image = self.mse_loss(pred_image, target_patches)
        mask_loss_image = mask_loss_image.mean(dim=-1)  # [B, num_patches]

        # Per-sample loss computation.
        sample_loss_sum = (mask_loss_image * image_mask_index).sum(dim=1)        # [B]
        sample_mask_count = image_mask_index.sum(dim=1)               # [B]
        valid_samples = sample_mask_count > 0             # [B]
        
        # Compute the average loss for each valid sample.
        sample_loss = sample_loss_sum / sample_mask_count.clamp(min=1e-8)  # Avoid division by zero.
        
        # Exclude invalid samples from the final average.
        mask_loss_image = (sample_loss[valid_samples].sum() / 
                    (valid_samples.sum() + 1e-8))  # +1e-8 avoids division by zero when all samples are invalid.

        return mask_loss_image
    
    def _compute_mask_loss_text(self, pred_text, target_text, text_mask_index):
        device = target_text.device
        loss_fn = nn.NLLLoss(reduction='none')  # Keep per-token losses instead of reducing immediately.

        # Compute per-token NLL loss.
        token_loss = loss_fn(pred_text.permute(0, 2, 1), target_text)  # Match pred_text shape to target_text.
        # Shape: [batch_size, 77].

        # Compute loss only for masked tokens.
        masked_loss = token_loss * text_mask_index.to(device)  # [batch_size, 77]

        # Compute per-sample average reconstruction loss over masked tokens.
        per_sample_loss = masked_loss.sum(dim=1) / text_mask_index.sum(dim=1).clamp(min=1).to(device)  # [batch_size]

        # Average only over samples that have at least one masked token.
        valid_samples = text_mask_index.sum(dim=1) > 0
        if valid_samples.any():
            return per_sample_loss[valid_samples].mean()
        else:
            return torch.tensor(0.0, device=device)
    
    def forward(self, model_output_dict):
        image_features = model_output_dict["img_feat_cls_token"]
        text_features = model_output_dict["text_feat_cls_token"]
        masked_image_features = model_output_dict["masked_image_feat_cls_token"]
        masked_text_features = model_output_dict["masked_text_feat_cls_token"]

        pred_image = model_output_dict["pred_image"]
        target_image = model_output_dict["target_image"]

        pred_text = model_output_dict["pred_text"]
        target_text = model_output_dict["target_text"]

        logit_scale = model_output_dict["logit_scale"]
        image_mask_index = model_output_dict["image_mask_index"]
        text_mask_index = model_output_dict["text_mask_index"]

        contrast_loss = self._compute_contrast_loss(image_features, text_features, masked_image_features, masked_text_features, logit_scale)
        intra_modal_contrast_loss = self._compute_intra_modal_contrast_loss(image_features, text_features, masked_image_features, masked_text_features, logit_scale)
        mask_loss_image = self._compute_mask_loss_image(pred_image, target_image, image_mask_index, self.patch_size)
        mask_loss_text = self._compute_mask_loss_text(pred_text, target_text, text_mask_index)

        total_loss = self.args.contrast_loss_w * contrast_loss + self.args.intra_modal_contrast_loss_w * intra_modal_contrast_loss + \
            self.args.mask_image_loss_w * mask_loss_image + self.args.mask_text_loss_w * mask_loss_text
        
        loss = {
            "contrast_loss": contrast_loss,
            "intra_modal_contrast_loss": intra_modal_contrast_loss,
            "mask_loss_text": mask_loss_text,
            "mask_loss_image": mask_loss_image,
            "total_loss": total_loss
        }
        
        return loss


## S06 · Training and checkpointing

The scheduler is copied from the pinned [author scheduler](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/training/scheduler.py). The single-GPU trainer follows the [author parameter groups and update path](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/src/training/main_train_macco_clip.py). It creates fresh model and optimizer state, uses AMP, includes the partial final batch, and saves its own checkpoint.


In [ ]:
import numpy as np


def assign_learning_rate(optimizer, clip_new_lr, predictor_new_lr):
    
    optimizer.param_groups[0]["lr"] = clip_new_lr
    optimizer.param_groups[1]["lr"] = clip_new_lr
    optimizer.param_groups[2]["lr"] = predictor_new_lr


def _warmup_lr(base_lr, warmup_length, step):
    return base_lr * (step + 1) / warmup_length


def cosine_lr(optimizer, clip_base_lr, predictor_base_lr, warmup_length, steps):
    def _lr_adjuster(step):
        if step < warmup_length:
            lr_clip = _warmup_lr(clip_base_lr, warmup_length, step)
            lr_predictor = _warmup_lr(predictor_base_lr, warmup_length, step)
        else:
            e = step - warmup_length
            es = steps - warmup_length
            lr_clip = 0.5 * (1 + np.cos(np.pi * e / es)) * clip_base_lr
            lr_predictor = 0.5 * (1 + np.cos(np.pi * e / es)) * predictor_base_lr
        assign_learning_rate(optimizer, lr_clip, lr_predictor)
        return lr_clip, lr_predictor
    return _lr_adjuster


def make_loader(rows_path, config, batch_size, shuffle=True):
    transform, get_params = create_transform_with_tracking(224)
    dataset = CsvDataset_mask_text_image(
        SimpleNamespace(**config), rows_path, transform, get_params, DATASETS
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=False, num_workers=config["workers"],
                      pin_memory=True, persistent_workers=config["workers"] > 0)

def make_optimizer(model, config):
    exclude = lambda name, p: p.ndim < 2 or any(
        key in name for key in ("bn", "ln", "bias", "logit_scale")
    )
    named = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
    clip_no_decay = [p for name, p in named if "CLIP" in name and exclude(name, p)]
    clip_decay = [p for name, p in named if "CLIP" in name and not exclude(name, p)]
    predictors = [p for name, p in named if "CLIP" not in name]
    assert clip_no_decay and clip_decay and predictors
    return torch.optim.AdamW([
        {"params": clip_no_decay, "weight_decay": 0.0, "lr": config["lr_clip"]},
        {"params": clip_decay, "weight_decay": config["wd"], "lr": config["lr_clip"]},
        {"params": predictors, "weight_decay": 0.01, "lr": config["lr_predictor"]},
    ], betas=(config["beta1"], config["beta2"]), eps=config["eps"])

def train_batch(model, loss_fn, optimizer, scaler, scheduler, batch, step):
    images, texts, _, text_mask, _, image_mask = batch
    images = images.to(DEVICE, non_blocking=True)
    texts = texts.to(DEVICE, non_blocking=True)
    text_mask = text_mask.to(DEVICE, non_blocking=True)
    image_mask = image_mask.to(DEVICE, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast("cuda", enabled=True):
        output = model(images, texts, image_mask, text_mask)
        terms = loss_fn(output)
    assert all(torch.isfinite(value).all().item() for value in terms.values()), "Nonfinite MACCO loss"
    previous_scale = scaler.get_scale()
    scaler.scale(terms["total_loss"]).backward()
    scheduler(step)
    scaler.step(optimizer)
    scaler.update()
    assert scaler.get_scale() >= previous_scale, "AMP skipped an update; lower initial scale and repeat the run"
    with torch.no_grad():
        model.logit_scale.clamp_(0, math.log(100))
    return {name: float(value.detach().cpu()) for name, value in terms.items()}, len(images)

def train_macco(config, train_rows, output_dir):
    config = dict(config)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    loader = make_loader(train_rows, config, config["batch_size"])
    assert len(loader.dataset) == 1000
    updates_per_epoch = len(loader)
    planned_updates = config["epochs"] * updates_per_epoch
    config["updates_per_epoch"] = updates_per_epoch
    config["planned_updates"] = planned_updates
    model = new_macco(config)
    model.train()
    optimizer = make_optimizer(model, config)
    scaler = torch.amp.GradScaler("cuda", enabled=True, init_scale=512)
    scheduler = cosine_lr(optimizer, config["lr_clip"], config["lr_predictor"],
                          config["warmup"], planned_updates)
    loss_fn = macco_clip_loss_text_image(SimpleNamespace(**config))
    logs = []
    step = 0
    for epoch in range(config["epochs"]):
        rows_seen = 0
        for batch in tqdm(loader, desc=f"Epoch {epoch+1}/{config['epochs']}"):
            values, count = train_batch(model, loss_fn, optimizer, scaler, scheduler, batch, step)
            logs.append({"epoch": epoch+1, "update": step+1, "batch_rows": count, **values})
            rows_seen += count
            step += 1
        assert rows_seen == 1000
        print(f"Epoch {epoch+1}: {rows_seen} rows, {step} cumulative updates")
        pd.DataFrame(logs).to_csv(output_dir / "training_log.csv", index=False)
    assert step == planned_updates
    checkpoint_path = output_dir / "final_checkpoint.pt"
    torch.save({"model": model.state_dict(), "config": config,
                "epoch": config["epochs"], "updates": step}, checkpoint_path)
    (output_dir / "resolved_config.yaml").write_text(
        yaml.safe_dump(config, sort_keys=False), encoding="utf-8"
    )
    del model, optimizer, scaler, loader
    gc.collect()
    torch.cuda.empty_cache()
    return {"checkpoint_path": checkpoint_path, "updates": step,
            "training_log": output_dir / "training_log.csv"}


## S07 · Shared evaluation

Frozen project-pool retrieval uses both directions at Recall@1/5/10. Selected Table 1 metrics are Sugar-Crepe Relation and Attribute; selected Table 21 metrics are Winoground Text/Image/Group and MMVP pair accuracy. Clean and noisy runs use identical cases and fixed per-image noise. Definitions follow the existing project evaluation notebooks and [author benchmark collection](https://huggingface.co/datasets/hiker-lw/VL-Compositionality-Benchmarks).


In [ ]:
EVAL_MEAN = torch.tensor((0.48145466, 0.4578275, 0.40821073))[:, None, None]
EVAL_STD = torch.tensor((0.26862954, 0.26130258, 0.27577711))[:, None, None]
eval_resize = torchvision.transforms.Resize(224, interpolation=torchvision.transforms.InterpolationMode.BICUBIC)
eval_crop = torchvision.transforms.CenterCrop(224)
eval_to_tensor = torchvision.transforms.ToTensor()

def prepare_eval_image(path, condition="clean", case_id=None):
    with Image.open(path) as source:
        image = eval_crop(eval_resize(source.convert("RGB")))
        pixels = eval_to_tensor(image)
    if condition == "noisy":
        key = f"{EVAL_CONFIG['seed']}:{case_id or path}"
        seed = int(hashlib.sha256(key.encode()).hexdigest()[:16], 16)
        generator = torch.Generator().manual_seed(seed)
        pixels = (pixels + EVAL_CONFIG["noise_sigma"] *
                  torch.randn(pixels.shape, generator=generator)).clamp(0, 1)
    elif condition != "clean":
        raise ValueError(condition)
    return (pixels - EVAL_MEAN) / EVAL_STD

@torch.inference_mode()
def score_retrieval(clip, rows, benchmark, split, condition, batch_size=32):
    rows = rows.reset_index(drop=True)
    assert rows.image_path.map(lambda path: Path(path).is_file()).all()
    image_vectors, text_vectors = [], []
    for first in tqdm(range(0, len(rows), batch_size), desc=f"{benchmark} {split} {condition}"):
        batch = rows.iloc[first:first+batch_size]
        ids = batch.image_id if "image_id" in batch else batch.command_token
        images = torch.stack([
            prepare_eval_image(path, condition, key)
            for path, key in zip(batch.image_path, ids)
        ]).to(DEVICE)
        texts = tokenize(batch.title.astype(str).tolist()).to(DEVICE)
        image_vectors.append(F.normalize(clip.encode_image(images).float(), dim=-1).cpu())
        text_vectors.append(F.normalize(clip.encode_text(texts).float(), dim=-1).cpu())
    image_vectors = torch.cat(image_vectors)
    text_vectors = torch.cat(text_vectors)
    similarities = image_vectors @ text_vectors.T
    records = []
    for direction, scores in [("image_to_text", similarities), ("text_to_image", similarities.T)]:
        order = scores.argsort(dim=1, descending=True)
        correct = torch.arange(len(rows))[:, None]
        for k in EVAL_CONFIG["retrieval_k"]:
            value = 100 * float((order[:, :min(k,len(rows))] == correct).any(dim=1).float().mean())
            records.append({
                "benchmark": benchmark, "split": split,
                "metric": f"{direction}_Recall@{k}", "condition": condition,
                "seed": EVAL_CONFIG["seed"], "n_examples": len(rows),
                "candidate_pool": len(rows), "score": value, "unit": "percent",
                "arm": "four_loss_baseline", "training_dataset": "COCO",
            })
    return records

BENCHMARK_REPO = "hiker-lw/VL-Compositionality-Benchmarks"
def extract_benchmark(name):
    marker = BENCHMARKS / ("." + name + ".done")
    if marker.exists():
        return
    url = hf_hub_url(BENCHMARK_REPO, name, repo_type="dataset")
    with requests.get(url, stream=True, timeout=(15, 180)) as response:
        response.raise_for_status()
        with tarfile.open(fileobj=response.raw, mode="r|*") as archive:
            for member in archive:
                archive.extract(member, BENCHMARKS, filter="data")
    marker.touch()

@lru_cache(None)
def image_index(root):
    index = {}
    for candidate in Path(root).rglob("*"):
        if candidate.is_file() and candidate.suffix.lower() in (".jpg", ".jpeg", ".png"):
            index.setdefault(candidate.name, []).append(candidate)
    return index

def find_image(root, relative):
    candidate = Path(root) / relative
    if candidate.is_file():
        return candidate
    matches = image_index(str(root)).get(Path(relative).name, [])
    assert len(matches) == 1, f"Missing or ambiguous image: {relative}"
    return matches[0]

def get_sugar_rows():
    extract_benchmark("sugar-crepe.tar.gz")
    sugar = next(BENCHMARKS.rglob("replace_rel.json")).parent
    groups = {}
    for name in ("replace_rel", "add_att", "replace_att", "swap_att"):
        annotations = json.loads((sugar / f"{name}.json").read_text(encoding="utf-8"))
        groups[name] = [
            (str(find_image(sugar / "val2017", entry["filename"])),
             entry["negative_caption"], entry["caption"])
            for entry in annotations.values()
        ]
    return groups

@torch.inference_mode()
def score_binary_captions(clip, rows, condition, batch_size=32):
    correct = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start+batch_size]
        images = torch.stack([
            prepare_eval_image(path, condition, path) for path, _, _ in batch
        ]).to(DEVICE)
        captions = [caption for _, negative, positive in batch
                    for caption in (negative, positive)]
        texts = tokenize(captions).to(DEVICE)
        image_vec = F.normalize(clip.encode_image(images).float(), dim=-1)
        text_vec = F.normalize(clip.encode_text(texts).float(), dim=-1).reshape(len(batch), 2, -1)
        scores = torch.einsum("bd,bkd->bk", image_vec, text_vec)
        correct.extend((scores[:, 1] > scores[:, 0]).cpu().tolist())
    return np.asarray(correct, dtype=bool)

CLEAN_IDS = {0, 1, 2, 5, 6, 7, 8, 9, 11, 12, 14, 15, 17, 18, 19, 20, 21, 24, 26, 29, 30, 32, 33, 34, 35, 37, 39, 43, 45, 47, 48, 50, 51, 52, 53, 54, 56, 57, 59, 60, 64, 66, 67, 71, 79, 80, 85, 87, 89, 90, 91, 92, 94, 98, 99, 100, 101, 102, 104, 105, 106, 107, 108, 109, 112, 115, 117, 120, 122, 123, 124, 125, 126, 127, 129, 137, 139, 140, 141, 142, 145, 146, 147, 151, 153, 154, 157, 158, 160, 161, 162, 165, 166, 167, 168, 169, 170, 171, 175, 177, 178, 179, 180, 181, 183, 184, 185, 186, 194, 195, 196, 197, 202, 205, 207, 212, 213, 216, 225, 231, 236, 240, 243, 244, 248, 250, 251, 252, 256, 259, 261, 265, 266, 269, 270, 271, 272, 273, 278, 279, 283, 285, 288, 289, 290, 291, 294, 297, 301, 302, 306, 308, 309, 317, 328, 337, 341, 349, 357, 360, 366, 368, 369, 370, 372, 378, 379, 380, 389, 391, 397}

def get_winoground_pairs():
    extract_benchmark("winoground.tar.gz")
    wino = next(BENCHMARKS.rglob("examples.jsonl")).parent
    if not (wino / "images").is_dir():
        with zipfile.ZipFile(next(wino.rglob("images.zip"))) as archive:
            archive.extractall(wino)
    examples = [json.loads(line) for line in (wino / "examples.jsonl").read_text().splitlines()]
    selected = [entry for entry in examples if entry["id"] in CLEAN_IDS]
    assert len(examples) == 400 and len(selected) == 171
    return [(wino / "images" / (entry["image_0"] + ".png"),
             wino / "images" / (entry["image_1"] + ".png"),
             entry["caption_0"], entry["caption_1"]) for entry in selected]

def get_mmvp_pairs():
    root = Path(snapshot_download("MMVP/MMVP_VLM", repo_type="dataset",
                                  allow_patterns=["Questions.csv", "MLLM_VLM Images/**"]))
    with (root / "Questions.csv").open(newline="") as handle:
        questions = list(csv.DictReader(handle))
    pairs = []
    for first, second in zip(questions[::2], questions[1::2]):
        a, b = int(first["Question ID"]), int(second["Question ID"])
        assert a % 2 == 1 and b == a + 1 and first["Type"] == second["Type"]
        folder = root / "MLLM_VLM Images" / first["Type"]
        pairs.append((folder / f"{a}.jpg", folder / f"{b}.jpg",
                      "a photo of " + first["Statement"],
                      "a photo of " + second["Statement"]))
    assert len(questions) == 270 and len(pairs) == 135
    return pairs

@torch.inference_mode()
def score_pairs(clip, pairs, condition, batch_size=16):
    blocks = []
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start+batch_size]
        images = torch.stack([
            prepare_eval_image(path, condition, path)
            for image0, image1, _, _ in batch for path in (image0, image1)
        ]).to(DEVICE)
        captions = [caption for _, _, caption0, caption1 in batch
                    for caption in (caption0, caption1)]
        texts = tokenize(captions).to(DEVICE)
        image_vec = F.normalize(clip.encode_image(images).float(), dim=-1)
        text_vec = F.normalize(clip.encode_text(texts).float(), dim=-1)
        blocks.append(torch.einsum("nkd,nld->nkl",
            image_vec.reshape(-1, 2, image_vec.shape[-1]),
            text_vec.reshape(-1, 2, text_vec.shape[-1])).cpu().numpy())
    return np.concatenate(blocks)

def metric_record(benchmark, metric, condition, n_examples, score):
    return {"benchmark": benchmark, "split": "paper_test", "metric": metric,
            "condition": condition, "seed": EVAL_CONFIG["seed"],
            "n_examples": n_examples, "candidate_pool": 2, "score": score,
            "unit": "percent", "arm": "four_loss_baseline", "training_dataset": "COCO"}

def score_table1(clip, groups, condition):
    values = {name: score_binary_captions(clip, rows, condition).mean()
              for name, rows in groups.items()}
    attribute = np.mean([values[name] for name in ("add_att", "replace_att", "swap_att")])
    return [
        metric_record("Sugar-Crepe", "Sugar-Crepe Relation", condition,
                      len(groups["replace_rel"]), 100*values["replace_rel"]),
        metric_record("Sugar-Crepe", "Sugar-Crepe Attribute", condition,
                      sum(len(groups[name]) for name in ("add_att", "replace_att", "swap_att")),
                      100*attribute),
    ]

def score_table21(clip, wino_pairs, mmvp_pairs, condition):
    wino = score_pairs(clip, wino_pairs, condition)
    text_right = (wino[:,0,0] > wino[:,0,1]) & (wino[:,1,1] > wino[:,1,0])
    image_right = (wino[:,0,0] > wino[:,1,0]) & (wino[:,1,1] > wino[:,0,1])
    mmvp = score_pairs(clip, mmvp_pairs, condition)
    mmvp_right = (mmvp[:,0,0] > mmvp[:,1,0]) & (mmvp[:,1,1] > mmvp[:,0,1])
    return [
        metric_record("Winoground", "Winoground Text", condition, len(wino_pairs), 100*text_right.mean()),
        metric_record("Winoground", "Winoground Image", condition, len(wino_pairs), 100*image_right.mean()),
        metric_record("Winoground", "Winoground Group", condition, len(wino_pairs),
                      100*(text_right & image_right).mean()),
        metric_record("MMVP-VLM", "MMVP Avg.", condition, len(mmvp_pairs), 100*mmvp_right.mean()),
    ]

def load_trained_clip(checkpoint_path):
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    clip = CLIP(embed_dim=512,
                vision_cfg=CLIPVisionCfg(layers=12, width=768, patch_size=32, image_size=224),
                text_cfg=CLIPTextCfg(context_length=77, vocab_size=49408,
                                     width=512, heads=8, layers=12),
                quick_gelu=True)
    state = {key.removeprefix("CLIP."): value
             for key, value in saved["model"].items() if key.startswith("CLIP.")}
    clip.load_state_dict(state, strict=True)
    return clip.float().to(DEVICE).eval()


## 00.02 · Check shared implementation

All shared definitions must be present before experiment settings or data are built.


In [ ]:
assert callable(tokenize) and callable(new_macco) and callable(train_macco)
assert callable(score_retrieval) and callable(score_table1) and callable(score_table21)
print("Pinned MACCO revision:", SOURCE_REV)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


## 00.03 · Common settings

Five epochs, author loss weights and learning rates, seed 0, AMP, and candidate batch 16. The speed check locks the final common size.


In [ ]:
BASELINE_CONFIG = {
    "source_revision": SOURCE_REV,
    "model": "ViT-B-32",
    "pretrained": "openai",
    "openai_checkpoint": CLIP_URL,
    "openai_checkpoint_sha256": CLIP_SHA256,
    "seed": 0,
    "epochs": 5,
    "batch_size_candidate": 16,
    "amp": True,
    "workers": 2,
    "warmup": 50,
    "lr_clip": 5e-7,
    "lr_predictor": 1e-3,
    "wd": 0.2,
    "beta1": 0.9,
    "beta2": 0.98,
    "eps": 1e-6,
    "text_predictor_self_attn_layers": 0,
    "text_predictor_cross_attn_layers": 2,
    "image_predictor_self_attn_layers": 0,
    "image_predictor_cross_attn_layers": 3,
    "mask_text_image": True,
    "mask_token_type": "relation_and_attribute",
    "random_rectangular_mask_prob": 0.1,
    "random_rectangular_mask_image_ratio": 0.75,
    "random_mask_image": False,
    "random_mask_text": False,
    "add_neg_sample": False,
    "contrast_loss_w": 1.0,
    "intra_modal_contrast_loss_w": 0.1,
    "mask_text_loss_w": 20.0,
    "mask_image_loss_w": 250.0,
}
EVAL_CONFIG = {
    "seed": 2026,
    "noise_sigma": 0.07,
    "project_pool_size": 500,
    "retrieval_k": [1, 5, 10],
    "table1_metrics": ["Sugar-Crepe Relation", "Sugar-Crepe Attribute"],
    "table21_metrics": ["Winoground Text", "Winoground Image", "Winoground Group", "MMVP Avg."],
    "conditions": ["clean", "noisy"],
}
print("Baseline settings:", yaml.safe_dump(BASELINE_CONFIG, sort_keys=False))


## 00.04 · Prepare COCO splits

Choose one author-annotated row per image: 1,000 train2014, 500 val2014 validation, and a disjoint 500 val2014 project test. If an approved-negative file is available in `/content/macco_inputs/`, prioritize its positive row IDs without excluding other selected positives. Stage the exact images from the official COCO host.


In [ ]:
TSV_PATH = download_asset(
    f"https://raw.githubusercontent.com/hiker-lw/MACCO/{SOURCE_REV}/mscoco_object_relation_attribute_with_phrase_bbox.tsv",
    DATA_ROOT / "mscoco_object_relation_attribute_with_phrase_bbox.tsv",
)
annotations = pd.read_csv(TSV_PATH, sep="\t")
required_columns = {
    "filepath", "title", "object_list", "relation_list", "attribute_list",
    "relation_phrase_list", "attribute_phrase_list",
    "relation_attribute_phrase_bbox_list", "neg_caption", "neg_image",
}
assert required_columns.issubset(annotations.columns), sorted(required_columns - set(annotations.columns))
annotations.insert(0, "row_id", np.arange(len(annotations)))
annotations["image_id"] = annotations.filepath.map(lambda value: Path(str(value)).stem)
train_all = annotations[annotations.filepath.str.contains("train2014", na=False)].drop_duplicates("image_id")
val_all = annotations[annotations.filepath.str.contains("val2014", na=False)].drop_duplicates("image_id")
assert len(train_all) >= 1000 and len(val_all) >= 1000

review_file = INPUT_ROOT / "cml_reviewed_negatives.csv"
preferred_ids = set()
if review_file.is_file():
    reviewed = pd.read_csv(review_file)
    if "approved" in reviewed:
        reviewed = reviewed[reviewed.approved.astype(str).str.lower().isin(["true", "1", "yes"])]
    if "row_id" in reviewed:
        preferred_ids = set(reviewed.row_id.astype(int))
    elif "image_id" in reviewed:
        preferred_ids = set(train_all[train_all.image_id.isin(reviewed.image_id.astype(str))].row_id)
priority = train_all[train_all.row_id.isin(preferred_ids)]
assert len(priority) <= 1000, "More than 1,000 approved positive image IDs; review the subset."
remaining = train_all[~train_all.image_id.isin(priority.image_id)]
coco_train = pd.concat([
    priority, remaining.sample(n=1000-len(priority), random_state=BASELINE_CONFIG["seed"])
]).reset_index(drop=True)
chosen_val = val_all.sample(n=1000, random_state=BASELINE_CONFIG["seed"]).reset_index(drop=True)
coco_val = chosen_val.iloc[:500].copy()
coco_test = chosen_val.iloc[500:].copy()
assert len(coco_train) == 1000 and len(coco_val) == len(coco_test) == 500
assert not set(coco_train.image_id) & set(coco_val.image_id)
assert not set(coco_train.image_id) & set(coco_test.image_id)
assert not set(coco_val.image_id) & set(coco_test.image_id)

def coco_destination(filepath):
    relative = Path(str(filepath))
    assert not relative.is_absolute() and ".." not in relative.parts
    return DATASETS / relative

def stage_coco(record):
    name = Path(str(record.filepath)).name
    split = "train2014" if "train2014" in str(record.filepath) else "val2014"
    assert name.startswith("COCO_" + split + "_")
    destination = coco_destination(record.filepath)
    download_asset(f"https://images.cocodataset.org/{split}/{name}", destination)
    with Image.open(destination) as image:
        image.verify()
    return str(destination)

all_coco = pd.concat([coco_train, coco_val, coco_test])
with ThreadPoolExecutor(max_workers=8) as executor:
    staged = list(tqdm(executor.map(stage_coco, all_coco.itertuples(index=False)),
                       total=len(all_coco), desc="Staging COCO images"))
for frame in (coco_train, coco_val, coco_test):
    frame["image_path"] = frame.filepath.map(lambda value: str(coco_destination(value)))
coco_train.to_csv(MANIFESTS / "coco_train.tsv", sep="\t", index=False)
coco_val.to_csv(MANIFESTS / "coco_val.csv", index=False)
coco_test.to_csv(MANIFESTS / "coco_test.csv", index=False)
print("COCO rows:", len(coco_train), len(coco_val), len(coco_test))
print("COCO unique image IDs:", *(part.image_id.nunique() for part in (coco_train, coco_val, coco_test)))
print("Reviewed positive IDs included:", len(priority))


## 00.05 · Prepare Talk2Car and evaluation splits

Join labeled Talk2Car-Expr to original Talk2Car records by command token. Keep one row per frame, reserve whole official training scenes for a 500-row project test, select 1,000 rows from other training scenes, and select 500 official validation rows. Stage selected images from the official Talk2Car archive. Freeze manifest hashes and evaluation settings.


In [ ]:
EXPR_BASE = "https://raw.githubusercontent.com/ThierryDeruyttere/Talk2Car-Expr/main/data/"
COMMAND_BASE = "https://raw.githubusercontent.com/talk2car/Talk2Car/master/data/commands/"
def read_json_asset(url, filename):
    path = download_asset(url, DATA_ROOT / filename)
    return json.loads(path.read_text(encoding="utf-8"))

expr_train = read_json_asset(EXPR_BASE + "talk2car_expr_train.json", "talk2car_expr_train.json")
expr_val = read_json_asset(EXPR_BASE + "talk2car_expr_val.json", "talk2car_expr_val.json")
commands_train = read_json_asset(COMMAND_BASE + "train_commands.json", "train_commands.json")["commands"]
commands_val = read_json_asset(COMMAND_BASE + "val_commands.json", "val_commands.json")["commands"]

def traffic_rows(commands, expressions, split):
    rows = []
    for command in commands:
        token = command["command_token"]
        entry = expressions.get(token)
        if not entry or not str(entry.get("description", "")).strip():
            continue
        box = entry.get("obj_box")
        if not box or len(box) != 4 or min(float(box[2]), float(box[3])) <= 0:
            continue
        image_name = Path(str(entry["img"])).name
        rows.append({
            "command_token": token,
            "scene_token": command["scene_token"],
            "sample_token": command["sample_token"],
            "image_name": image_name,
            "image_path": str(DATA_ROOT / "traffic_images" / image_name),
            "title": str(entry["description"]).strip(),
            "box_xyxy": json.dumps([float(box[0]), float(box[1]),
                                    float(box[0])+float(box[2]), float(box[1])+float(box[3])]),
            "source_split": split,
        })
    frame = pd.DataFrame(rows).sort_values("command_token")
    return frame.drop_duplicates("sample_token").reset_index(drop=True)

traffic_train_all = traffic_rows(commands_train, expr_train, "train")
traffic_val_all = traffic_rows(commands_val, expr_val, "val")
assert len(traffic_train_all) >= 1500 and len(traffic_val_all) >= 500
rng = random.Random(BASELINE_CONFIG["seed"])
scenes = sorted(traffic_train_all.scene_token.unique())
rng.shuffle(scenes)
test_scenes = set()
for scene in scenes:
    test_scenes.add(scene)
    if traffic_train_all.scene_token.isin(test_scenes).sum() >= 500:
        break
test_pool = traffic_train_all[traffic_train_all.scene_token.isin(test_scenes)]
train_pool = traffic_train_all[~traffic_train_all.scene_token.isin(test_scenes)]
assert len(train_pool) >= 1000
traffic_test = test_pool.sample(n=500, random_state=BASELINE_CONFIG["seed"]).reset_index(drop=True)
traffic_train = train_pool.sample(n=1000, random_state=BASELINE_CONFIG["seed"]).reset_index(drop=True)
traffic_val = traffic_val_all.sample(n=500, random_state=BASELINE_CONFIG["seed"]).reset_index(drop=True)
for left, right in [(traffic_train, traffic_val), (traffic_train, traffic_test), (traffic_val, traffic_test)]:
    assert not set(left.scene_token) & set(right.scene_token)
    assert not set(left.sample_token) & set(right.sample_token)
assert all(len(frame) == count for frame, count in
           [(traffic_train, 1000), (traffic_val, 500), (traffic_test, 500)])

for frame, name in [(traffic_train, "traffic_train.csv"),
                    (traffic_val, "traffic_val.csv"),
                    (traffic_test, "traffic_test.csv")]:
    frame.to_csv(MANIFESTS / name, index=False)

traffic_images = DATA_ROOT / "traffic_images"
traffic_images.mkdir(exist_ok=True)
needed = set(pd.concat([traffic_train, traffic_val, traffic_test]).image_name)
missing = {name for name in needed if not (traffic_images / name).is_file()}
if missing:
    import gdown
    image_zip = INPUT_ROOT / "imgs.zip"
    if not image_zip.is_file():
        result = gdown.download(id="1bhcdej7IFj5GqfvXGrHGPk2Knxe77pek",
                                output=str(image_zip), quiet=False)
        assert result and image_zip.is_file(), "Official Talk2Car image archive download failed."
    with zipfile.ZipFile(image_zip) as archive:
        archive_lookup = {}
        for info in archive.infolist():
            if info.is_dir():
                continue
            filename = Path(info.filename).name
            corrected = filename[4:] if filename.startswith(("img_train_", "img_val_")) else filename
            if corrected in missing:
                archive_lookup[corrected] = info
        assert set(archive_lookup) == missing, f"Missing traffic images: {sorted(missing - set(archive_lookup))[:10]}"
        for name, info in tqdm(archive_lookup.items(), desc="Staging Talk2Car images"):
            with archive.open(info) as source, (traffic_images / name).open("wb") as output:
                shutil.copyfileobj(source, output)
assert all((traffic_images / name).is_file() for name in needed)
print("Talk2Car rows:", len(traffic_train), len(traffic_val), len(traffic_test))
print("Talk2Car unique scenes:", *(part.scene_token.nunique() for part in
                                      (traffic_train, traffic_val, traffic_test)))


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

EVAL_CONFIG["manifests"] = {
    filename: {"sha256": file_sha256(MANIFESTS / filename)}
    for filename in ("coco_val.csv", "coco_test.csv", "traffic_val.csv", "traffic_test.csv")
}
EVAL_CONFIG["talk2car_project_test_source"] = "official labeled train scenes reserved whole"
(MANIFESTS / "eval_config.yaml").write_text(
    yaml.safe_dump(EVAL_CONFIG, sort_keys=False), encoding="utf-8"
)
print("Frozen evaluation manifest hashes:")
for name, item in EVAL_CONFIG["manifests"].items():
    print(name, item["sha256"])


## 00.06 · Smoke test and batch-speed check

Check one four-loss optimizer update, finite terms, original CLIP initialization, and checkpoint reload at batch 8. Benchmark 8, 16, and 32 with five untimed and 20 timed updates per size. The CML, pixel-noise, and traffic probes check the selected memory footprint; the fastest stable candidate with headroom becomes the shared batch size.


In [ ]:
SMOKE_CONFIG = dict(BASELINE_CONFIG, batch_size=8)
smoke_loader = make_loader(MANIFESTS / "coco_train.tsv", SMOKE_CONFIG, 8)
fixed_batch = next(iter(smoke_loader))
smoke_model = new_macco(SMOKE_CONFIG)
smoke_model.train()
smoke_loss = macco_clip_loss_text_image(SimpleNamespace(**SMOKE_CONFIG))
smoke_optimizer = make_optimizer(smoke_model, SMOKE_CONFIG)
smoke_scaler = torch.amp.GradScaler("cuda", init_scale=512)
smoke_scheduler = cosine_lr(smoke_optimizer, SMOKE_CONFIG["lr_clip"],
                            SMOKE_CONFIG["lr_predictor"], 50, 100)
before = smoke_model.CLIP.visual.conv1.weight.detach().clone()
smoke_terms, smoke_count = train_batch(
    smoke_model, smoke_loss, smoke_optimizer, smoke_scaler, smoke_scheduler,
    fixed_batch, 0
)
after = smoke_model.CLIP.visual.conv1.weight.detach().cpu().clone()
assert smoke_model.CLIP.visual.conv1.weight.grad is not None
assert smoke_model.CLIP.token_embedding.weight.grad is not None
assert int(smoke_optimizer.state[smoke_model.CLIP.visual.conv1.weight]["step"]) == 1
assert int(smoke_optimizer.state[smoke_model.CLIP.token_embedding.weight]["step"]) == 1
assert all(math.isfinite(value) for value in smoke_terms.values())
assert math.isclose(smoke_terms["total_loss"],
    smoke_terms["contrast_loss"] + 0.1*smoke_terms["intra_modal_contrast_loss"]
    + 20*smoke_terms["mask_loss_text"] + 250*smoke_terms["mask_loss_image"],
    rel_tol=1e-4, abs_tol=1e-4)
smoke_path = RUN_ROOT / "smoke_checkpoint.pt"
torch.save(smoke_model.state_dict(), smoke_path)
del smoke_model, smoke_optimizer, smoke_scaler
gc.collect()
torch.cuda.empty_cache()
reloaded = new_macco(SMOKE_CONFIG)
reloaded.load_state_dict(torch.load(smoke_path, map_location=DEVICE, weights_only=True))
assert torch.equal(reloaded.CLIP.visual.conv1.weight.cpu(), after)
print("Smoke test:", smoke_count, "rows, finite four-loss update, checkpoint reload passed")
print(smoke_terms)
del reloaded, smoke_loader, fixed_batch
gc.collect()
torch.cuda.empty_cache()
smoke_path.unlink(missing_ok=True)


In [ ]:
def repeat_batches(loader):
    while True:
        yield from loader

def variant_fit_check(model, loss_fn, optimizer, scaler, batch):
    images, texts, _, text_mask, _, image_mask = batch
    images = images.to(DEVICE, non_blocking=True)
    texts = texts.to(DEVICE, non_blocking=True)
    text_mask = text_mask.to(DEVICE, non_blocking=True)
    image_mask = image_mask.to(DEVICE, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast("cuda"):
        output = model(images, texts, image_mask, text_mask)
        negative, _, _, _ = model.CLIP.encode_text(
            texts, torch.zeros_like(text_mask), model.text_mask_token
        )
        cml_probe = F.relu(0.05 + F.cosine_similarity(output["img_feat_cls_token"], negative)
                           - F.cosine_similarity(output["img_feat_cls_token"],
                                                 output["text_feat_cls_token"])).mean()
        total = loss_fn(output)["total_loss"] + cml_probe
    scaler.scale(total).backward()
    optimizer.zero_grad(set_to_none=True)
    # Pixel noise and Talk2Car have the same image tensor shape and batch size.
    noisy = images + 0.01*torch.randn_like(images)
    with torch.amp.autocast("cuda"):
        noisy_total = loss_fn(model(noisy, texts, image_mask, text_mask))["total_loss"]
    scaler.scale(noisy_total).backward()
    optimizer.zero_grad(set_to_none=True)
    traffic_batch = traffic_train.head(len(images))
    traffic_transform, _ = create_transform_with_tracking(224)
    traffic_images = torch.stack([
        traffic_transform(Image.open(path).convert("RGB"))
        for path in traffic_batch.image_path
    ]).to(DEVICE)
    traffic_texts = tokenize(traffic_batch.title.tolist()).to(DEVICE)
    traffic_image_mask = torch.zeros(len(traffic_batch), 50, dtype=torch.bool, device=DEVICE)
    traffic_image_mask[:, 1] = True
    traffic_text_mask = torch.zeros(len(traffic_batch), 77, dtype=torch.bool, device=DEVICE)
    traffic_text_mask[:, 1] = True
    with torch.amp.autocast("cuda"):
        traffic_total = loss_fn(model(traffic_images, traffic_texts,
                                     traffic_image_mask, traffic_text_mask))["total_loss"]
    scaler.scale(traffic_total).backward()
    optimizer.zero_grad(set_to_none=True)

speed_rows = []
gpu_total = torch.cuda.get_device_properties(0).total_memory
for batch_size in (16, 8, 32):
    torch.cuda.empty_cache()
    try:
        config = dict(BASELINE_CONFIG, batch_size=batch_size)
        loader = make_loader(MANIFESTS / "coco_train.tsv", config, batch_size)
        stream = repeat_batches(loader)
        model = new_macco(config)
        model.train()
        loss_fn = macco_clip_loss_text_image(SimpleNamespace(**config))
        optimizer = make_optimizer(model, config)
        scaler = torch.amp.GradScaler("cuda", init_scale=512)
        scheduler = cosine_lr(optimizer, config["lr_clip"], config["lr_predictor"],
                              50, 5 * len(loader))
        torch.cuda.reset_peak_memory_stats()
        for step in range(5):
            train_batch(model, loss_fn, optimizer, scaler, scheduler, next(stream), step)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for step in range(5, 25):
            train_batch(model, loss_fn, optimizer, scaler, scheduler, next(stream), step)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        variant_fit_check(model, loss_fn, optimizer, scaler, next(stream))
        peak = torch.cuda.max_memory_allocated()
        speed_rows.append({"gpu": torch.cuda.get_device_name(0), "batch_size": batch_size,
                           "images_per_second": 20*batch_size/elapsed,
                           "peak_memory_gb": peak/1e9, "gpu_memory_gb": gpu_total/1e9,
                           "completed": True})
    except (RuntimeError, AssertionError) as error:
        speed_rows.append({"gpu": torch.cuda.get_device_name(0), "batch_size": batch_size,
                           "images_per_second": np.nan, "peak_memory_gb": np.nan,
                           "gpu_memory_gb": gpu_total/1e9, "completed": False,
                           "error": str(error)[:200]})
        print(f"Batch {batch_size} failed:", error)
    finally:
        for name in ("model", "optimizer", "scaler", "loader", "loss_fn", "scheduler", "stream"):
            if name in locals():
                del globals()[name]
        gc.collect()
        torch.cuda.empty_cache()

speed = pd.DataFrame(speed_rows)
speed.to_csv(MANIFESTS / "batch_speed_check.csv", index=False)
eligible = speed[(speed.completed) & (speed.peak_memory_gb < 0.90*speed.gpu_memory_gb)]
assert len(eligible), "No candidate passed the speed and memory checks"
BATCH_SIZE = int(eligible.sort_values("images_per_second", ascending=False).iloc[0].batch_size)
BASELINE_CONFIG["batch_size"] = BATCH_SIZE
print(speed.to_string(index=False))
print("Locked common batch size:", BATCH_SIZE)


## 00.07 · Smoke test shared evaluation

Exercise the fixed COCO, Talk2Car, Sugar-Crepe, Winoground, and MMVP paths before training. These smoke scores are not baseline results.


In [ ]:
plain_clip = load_original_clip().to(DEVICE).eval()
for name, rows in [("COCO", coco_val.head(8)), ("Talk2Car", traffic_val.head(8))]:
    records = score_retrieval(plain_clip, rows, name, "validation_smoke", "clean")
    assert records and all(record["n_examples"] == 8 for record in records)
sugar_groups = get_sugar_rows()
winoground_pairs = get_winoground_pairs()
mmvp_pairs = get_mmvp_pairs()
assert len(score_binary_captions(plain_clip, sugar_groups["replace_rel"][:2], "clean")) == 2
assert score_pairs(plain_clip, winoground_pairs[:2], "clean").shape == (2, 2, 2)
assert score_pairs(plain_clip, mmvp_pairs[:2], "clean").shape == (2, 2, 2)
print("Shared evaluation smoke test passed on COCO, Talk2Car, Sugar-Crepe, Winoground, and MMVP.")
del plain_clip
gc.collect()
torch.cuda.empty_cache()


## 00.08 · Train baseline

Start again from the original OpenAI checkpoint and fresh predictors and optimizer. Train on the frozen 1,000 COCO pairs for five full epochs. Save per-term loss logs, resolved configuration, update count, and checkpoint.


In [ ]:
(MANIFESTS / "shared_run_config.yaml").write_text(
    yaml.safe_dump(BASELINE_CONFIG, sort_keys=False), encoding="utf-8"
)
baseline_result = train_macco(
    config=BASELINE_CONFIG,
    train_rows=MANIFESTS / "coco_train.tsv",
    output_dir=RUN_ROOT / "baseline",
)
print("Baseline checkpoint:", baseline_result["checkpoint_path"])
print("Optimizer updates:", baseline_result["updates"])
assert baseline_result["updates"] == 5 * math.ceil(1000 / BATCH_SIZE)


## 00.09 · Evaluate baseline

Evaluate the trained plain CLIP encoders on the frozen project pools and selected Table 1/21 metrics in clean and fixed-noisy conditions. No test score is recorded until training has completed.


In [ ]:
clip = load_trained_clip(baseline_result["checkpoint_path"])
results = []
for condition in EVAL_CONFIG["conditions"]:
    for benchmark, split, rows in [
        ("COCO", "validation", coco_val),
        ("COCO", "project_test", coco_test),
        ("Talk2Car", "validation", traffic_val),
        ("Talk2Car", "project_test", traffic_test),
    ]:
        results.extend(score_retrieval(clip, rows, benchmark, split, condition))
    results.extend(score_table1(clip, sugar_groups, condition))
    results.extend(score_table21(clip, winoground_pairs, mmvp_pairs, condition))
baseline_metrics = pd.DataFrame(results)
keys = ["benchmark", "split", "metric", "condition", "seed"]
assert not baseline_metrics.duplicated(keys).any()
assert baseline_metrics.score.notna().all()
baseline_metrics.to_csv(RUN_ROOT / "baseline" / "baseline_metrics.csv", index=False)
print(baseline_metrics[["benchmark","split","metric","condition","n_examples","score"]].to_string(index=False))
del clip
gc.collect()
torch.cuda.empty_cache()


## 00.10 · Save baseline handoff

Copy manifests, speed measurements, settings, training log, and baseline scores to Google Drive. Verify copied hashes, then remove the runtime checkpoint after its planned evaluations.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
EXPORT_ROOT = Path("/content/drive/MyDrive/MACCO_independent/baseline")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
handoff_files = list(MANIFESTS.glob("*.csv")) + list(MANIFESTS.glob("*.tsv")) + list(MANIFESTS.glob("*.yaml"))
handoff_files += [
    RUN_ROOT / "baseline" / "training_log.csv",
    RUN_ROOT / "baseline" / "resolved_config.yaml",
    RUN_ROOT / "baseline" / "baseline_metrics.csv",
]
if review_file.is_file():
    handoff_files.append(review_file)
for source in handoff_files:
    destination = EXPORT_ROOT / source.name
    shutil.copy2(source, destination)
    assert file_sha256(source) == file_sha256(destination)
print("Saved handoff files to:", EXPORT_ROOT)
for item in sorted(EXPORT_ROOT.iterdir()):
    print(item.name, item.stat().st_size, "bytes")
baseline_result["checkpoint_path"].unlink()
print("Removed runtime checkpoint after saving final scores, settings, and logs to Drive.")
